# MU Student Support Chatbot — LLM-only vs RAG (v3, new trilingual dataset)

**Thesis:** Development and Evaluation of an AI-Powered Student Support Chatbot Using a Pre-trained Language Model (Metropolitan University, Sylhet)

This notebook compares a pre-trained LLM answering on its own with the same LLM inside a Retrieval-Augmented Generation (RAG) pipeline. All three use the new MU dataset (1,445 verified facts, 7,169 questions in English, Bangla and Banglish).

| System | What it does |
|---|---|
| **S1. LLM-only** | Qwen2.5-7B-Instruct (4-bit) answers from its own knowledge. It is told to say "I don't know" when unsure. |
| **S2. RAG baseline** | Dense bge-m3 retrieval only, with a cosine threshold. This is the design of the previous notebook. |
| **S3. RAG final (this work)** | The full pipeline (see below). The answer is the verified dataset answer, optionally rewritten by the same LLM behind a grounding guard. |

**Final pipeline:** language detection (en / bn / Banglish) → **entity check** (batch `58 A`, course code `CSE 0613 1307`, `bus 03`). An unknown entity is refused, and a known one restricts retrieval to facts that contain exactly that entity. A question about another university (BRAC, SUST, ...) is refused, never answered with MU facts. Next comes **hybrid search**: bge-m3 dense + BM25 + character n-grams, fused with RRF, with one score per *fact*, so the three language copies of an answer never fill the top-k. Then a **cross-encoder reranker** (bge-reranker-v2-m3) orders the candidates. The **refusal threshold and margin are tuned on the dev split**. Finally the system answers in the user's own language, or **refuses with a routed message**: Exam Office for exam topics, MU office for other MU topics, out-of-domain for everything else.

**Evaluation data:** `MUBot_v5_EvalQuestions.csv` contains **new questions that are not in the knowledge base**: paraphrases, hard near-duplicates (`58 A` vs `58 B`, `CSE 0613 1307` vs `1308`, bus 01 vs 02), out-of-domain questions and unknown entities. It is split into **25 % dev** (used only for tuning) and **75 % test** (used once). A typo-noised copy of the test set measures robustness.

**Kaggle setup:** Accelerator = **GPU T4 x2**, Internet = **On**. Add a dataset that contains `MUBot_v5_KnowledgeBase.csv`, `MUBot_v5_Facts.csv` and `MUBot_v5_EvalQuestions.csv`, then *Run all*. A full run takes roughly 45–70 minutes, mostly LLM generation. For a quick trial (about 15 minutes) set `QUICK_TEST = True` in the first lines of Section 1. **Use the full run (`QUICK_TEST = False`) for the thesis results.**

## Google Colab version
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run the upload cell below and choose the 3 CSV files: `MUBot_v5_KnowledgeBase.csv`, `MUBot_v5_Facts.csv`, `MUBot_v5_EvalQuestions.csv`.
3. Run the remaining cells in order. Everything runs on one T4 GPU.
4. For the web app: click the 🔑 **Secrets** icon on the left, add `NGROK_TOKEN`, and turn on *Notebook access*.

In [ ]:
# Step 1: install the extra packages (takes about 1 minute)
!pip install -q rank_bm25 rapidfuzz bitsandbytes accelerate langchain-core

## 1. Configuration

In [ ]:
# ---- imports (kept in this cell so it works even if the kernel was restarted after the install cell) ----
import os, glob, json, re, random, time, gc, math, importlib
from collections import Counter
import numpy as np, pandas as pd
try:
    import torch
except ImportError:          # only for offline unit tests
    torch = None
pd.set_option("display.max_colwidth", 120)
if torch is not None and torch.cuda.is_available():
    print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
else:
    print("WARNING: no GPU found. In Kaggle: Settings -> Accelerator -> GPU T4 x2")

# =====================================================================================
# QUICK_TEST = True  -> quick trial on 120 test questions (about 15 min) to check that everything works
# QUICK_TEST = False -> full evaluation on all test questions (about 45-70 min) -> use these results in the thesis
QUICK_TEST = False
# =====================================================================================
SEED = 42
EMBED_MODEL  = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
LLM_MODEL    = "Qwen/Qwen2.5-7B-Instruct"      # 4-bit; falls back to Qwen2.5-3B automatically if 4-bit loading fails

N_RERANK       = 20      # candidates sent to the reranker
WRONG_PENALTY  = 3.0     # threshold tuning: one wrong answer costs as much as 3 refused correct answers
DIRECT_SIM     = 0.90    # a question almost identical to a dataset question (bge-m3 cosine >= 0.90) is always answered
USE_LLM_REWRITE = True   # RAG: let the LLM rephrase short verified answers (guarded); False = verified answer only
REWRITE_MAX_CHARS = 350  # longer verified answers (lists, routines) are returned verbatim
GEN_BATCH = 8            # questions per LLM forward pass (lower to 4 if CUDA out of memory)
MAX_NEW_TOKENS = 200
MAX_EVAL = 120 if QUICK_TEST else None   # number of test questions (None = all)
MOCK = os.environ.get("MU_MOCK") == "1"        # offline unit test only - leave unset on Kaggle

N_GPU = torch.cuda.device_count() if (torch is not None and torch.cuda.is_available()) else 0
DEV_LLM = "cuda:0" if N_GPU else "cpu"
DEV_RET = "cuda:1" if N_GPU > 1 else DEV_LLM
random.seed(SEED); np.random.seed(SEED)
if torch is not None: torch.manual_seed(SEED)
os.makedirs("outputs", exist_ok=True)
print("LLM on", DEV_LLM, "| retrieval models on", DEV_RET, "| MOCK =", MOCK)

## 2. Data: knowledge base, facts and the separate evaluation questions
Nothing is trained in this notebook: all models are used as pre-trained models. The **knowledge base** is the whole dataset (7,169 questions → 1,445 facts), and the chatbot searches it. The evaluation questions are separate new questions:
* **dev** (25 % of the in-domain questions, 40 % of the out-of-domain ones) works like a *validation set*. It is used only to choose the refusal threshold and the final retriever.
* **test** (the rest) is used once, at the end, for the reported results.

In [ ]:
# Upload the 3 CSV files (skip if they are already in /content)
import os
need = ["MUBot_v5_KnowledgeBase.csv", "MUBot_v5_Facts.csv", "MUBot_v5_EvalQuestions.csv"]
if not all(os.path.exists(f) for f in need):
    from google.colab import files
    up = files.upload()
    for name in list(up):                 # Colab renames repeated uploads to "name (1).csv"
        for n in need:
            if name != n and name.startswith(n[:-4]):
                os.replace(name, n)
print({f: os.path.exists(f) for f in need})

In [ ]:
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(name) + glob.glob(f"**/{name}", recursive=True)
    if not hits: raise FileNotFoundError(f"{name} not found - add the dataset with the 3 CSV files")
    return hits[0]

kb    = pd.read_csv(find("MUBot_v5_KnowledgeBase.csv"), keep_default_na=False)
facts = pd.read_csv(find("MUBot_v5_Facts.csv"), keep_default_na=False)
ev    = pd.read_csv(find("MUBot_v5_EvalQuestions.csv"), keep_default_na=False)
ev["gold"] = ev["gold_facts"].map(lambda s: [x for x in str(s).split("|") if x])
FACT = facts.set_index("fact_id")

# leakage checks: no evaluation question is in the knowledge base, every gold fact exists
assert not set(ev["question"].str.lower().str.strip()) & set(kb["question"].str.lower().str.strip())
assert all(g in FACT.index for gs in ev["gold"] for g in gs)

dev, test = ev[ev.split == "dev"].reset_index(drop=True), ev[ev.split == "test"].reset_index(drop=True)
if MAX_EVAL and MAX_EVAL < len(test):     # stratified sample: same share of in-domain / out-of-domain questions
    parts = [g.sample(min(len(g), max(10, int(MAX_EVAL * len(g) / len(test)))), random_state=SEED)
             for _, g in test.groupby("expected")]
    test = pd.concat(parts).reset_index(drop=True)
print(f"Knowledge base: {len(kb)} questions -> {len(facts)} facts | languages {kb.language.value_counts().to_dict()}")
print(f"Eval questions: dev {len(dev)} | test {len(test)}")
display(ev.groupby(["split", "type"]).size().unstack(0))
display(ev.groupby(["split", "language"]).size().unstack(0))

## 3. Retrieval module (`mu_rag.py`, also used by the web app)

In [ ]:
%%writefile mu_rag.py
"""mu_rag.py - retrieval, refusal decision and grounded answering for the Metropolitan University chatbot.

Used by the evaluation notebook and by the Streamlit app, so both run exactly the same pipeline.

Pipeline (final system):
  query -> language detection (en / bn / banglish)
        -> entity check (batch "58 A", course code "CSE 0613 1307", bus "bus 03"):
             unknown entity  -> refuse (never guess a neighbouring batch / code)
             known entity    -> only facts that contain that exact entity can be retrieved
        -> hybrid retrieval: bge-m3 dense (questions + answers) + BM25 (questions + answers) + char n-grams, RRF fusion
           (one score per FACT, so the 3 language copies of the same answer never fill the top-k)
        -> cross-encoder reranker (bge-reranker-v2-m3) on the top candidates
        -> decision: answer only if reranker score >= tau AND margin to the 2nd fact >= mu (both tuned on dev)
        -> answer = verified dataset answer in the user's language; optionally rewritten by the LLM,
           but the rewrite is used only if it passes the grounding guard (else the verified answer is returned)
        -> refusals are routed: exam topics -> Exam Office, other MU topics -> MU office, everything else -> out of domain
"""
import re
import math
from collections import Counter, defaultdict

import numpy as np
from rank_bm25 import BM25Okapi
from rapidfuzz import process, fuzz
from sklearn.feature_extraction.text import TfidfVectorizer

BN_RE = re.compile(r"[ঀ-৿]")
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
LANG_COL = {"en": "answer_en", "bn": "answer_bn", "banglish": "answer_banglish"}


# ----------------------------------------------------------------------------- text helpers
def _stem(w):
    # light English plural stripping ("students" -> "student"), used for BM25 on both the questions and the query
    return w[:-1] if len(w) > 3 and w.isalpha() and w.isascii() and w.endswith("s") and not w.endswith("ss") else w


def tokens(text):
    return [_stem(w) for w in re.findall(r"[a-z0-9]+|[ঀ-৿]+", str(text).lower().translate(BN_DIGITS))]


def _norm_num(s):
    if "." in s:
        s = s.rstrip("0").rstrip(".")
    return s.lstrip("0") or "0"


def numbers(text):
    t = str(text).translate(BN_DIGITS)
    t = re.sub(r"(?<=\d),(?=\d{3})", "", t)
    return {_norm_num(x) for x in re.findall(r"\d+(?:\.\d+)?", t)}


EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")


def emails(text):
    return {e.lower().rstrip(".") for e in EMAIL_RE.findall(str(text))}


def bn_ratio(text):
    letters = re.findall(r"[A-Za-z\u0980-\u09FF]", str(text))
    if not letters:
        return 0.0
    return sum(bool(BN_RE.match(c)) for c in letters) / len(letters)


def script_ok(text, lang, ref=None):
    """Bangla replies must contain real Bengali script (unless the reference itself is almost all Latin:
    codes, names, emails); English / Banglish replies must be in Latin script."""
    r = bn_ratio(text)
    if lang == "bn":
        if ref is not None and bn_ratio(ref) < 0.08:
            return True
        return r >= 0.08
    return r < 0.08


REFUSAL_RE = re.compile(
    r"(i don't know|i do not know|i don't have|i do not have|don't have (that|this|any|the)|do not have (that|this|any|the)|"
    r"not sure|no information|not available|unable to|cannot (provide|find|answer|access|help)|can't (provide|find|answer|access|help)|"
    r"i'm sorry|i am sorry|sorry|"
    r"জানি না|জানা নেই|তথ্য নেই|আমার কাছে নেই|দুঃখিত|নিশ্চিত নই|"
    r"jani na|jana nei|tottho nei|amar kache nei|dukkhito|sure na)", re.I)


def looks_like_refusal(text):
    return bool(REFUSAL_RE.search(str(text)))


NEG_RE = re.compile(r"\b(no|not|don't|doesn't|does not|isn't|aren't|cannot|can't|never|nei|nai|na)\b|নেই|না\b|নাই", re.I)
YES_START = re.compile(r"^\s*(yes|হ্যাঁ|hae|ha|hya)\b", re.I)
NO_START = re.compile(r"^\s*(no|না|na)\b[.,!।]", re.I)


def polarity(ref):
    if YES_START.search(ref):
        return "yes"
    if NO_START.search(ref):
        return "no"
    return None


CONSENSUS_MAX_CONF = 0.5     # below this reranker score, dense + hybrid agreement may choose the fact (see retrieve)

# ----------------------------------------------------------------------------- query normalisation
CANON = {
    "kto": "koto", "kot": "koto", "ase": "ache", "achhe": "ache", "asey": "ache", "acha": "ache", "ase?": "ache",
    "kothai": "kothay", "kothae": "kothay", "koi": "kothay", "kibhabe": "kivabe", "kemne": "kivabe", "kmne": "kivabe",
    "koita": "koyta", "koiti": "koyta", "koyti": "koyta", "kunta": "konta", "kun": "kon",
    "fi": "fee", "fis": "fee", "fees": "fee", "feez": "fee", "rezalt": "result", "rejalt": "result",
    "librery": "library", "libary": "library", "lybrary": "library", "scolarship": "scholarship",
    "scholership": "scholarship", "pasword": "password", "passwrd": "password", "pass": "password",
    "wai": "wifi", "wi": "wifi", "bas": "bus", "varsity": "university", "versity": "university",
    "xm": "exam", "exm": "exam", "stuff": "staff", "stuf": "staff", "studen": "student", "supplies": "supply", "supplie": "supply", "supplys": "supply", "supples": "supply", "supple": "supply", "eksam": "exam", "rejistration": "registration", "registation": "registration",
    "sir": "", "mam": "", "madam": "", "plz": "", "pls": "", "please": "", "vai": "", "apu": "", "bhai": "",
}
# extra tokens added (not replacing) so both spellings can match the knowledge base
EXPAND = {
    "supply": ["supplementary"], "supli": ["supply", "supplementary"], "sapli": ["supply", "supplementary"],
    "suply": ["supply", "supplementary"], "shapli": ["supply", "supplementary"], "supplimentary": ["supplementary", "supply"],
    "retek": ["retake"], "ritek": ["retake"], "rtk": ["retake"],
    "porikkha": ["exam"], "porikkhar": ["exam"], "porikkhay": ["exam"], "exam": ["porikkha"],
    "vorti": ["admission", "bhorti"], "bhorti": ["admission"], "admission": ["bhorti"],
    "teacher": ["faculty"], "faculty": ["teacher"], "mu": ["metropolitan", "university"], "sathe": ["eksathe"], "ek": ["eksathe"],
}
# Banglish -> rough English gloss, used only as an EXTRA query variant for the dense model and the reranker
GLOSS = {"ki": "what", "kivabe": "how", "kibhabe": "how", "kemne": "how", "koto": "how much", "kobe": "when",
         "kokhon": "when", "kothay": "where", "ke": "who", "kara": "who", "koyta": "how many", "koyjon": "how many",
         "kotojon": "how many", "ache": "is there", "nai": "no", "nei": "no", "er": "of", "theke": "from", "dite": "pay",
         "korte": "do", "korbo": "do", "lagbe": "need", "lage": "need", "jonno": "for", "kon": "which", "konta": "which",
         "porikkha": "exam", "porikkhar": "exam", "vorti": "admission", "bhorti": "admission", "deya": "give",
         "jay": "can", "parbo": "can", "pabo": "get", "shuru": "start", "shesh": "end", "chuti": "holiday", "hoy": ""}
PHRASES = [(r"\bek\s*s?h?athe\b", "eksathe"), (r"\bek\s*(time|somoy|bar)\s*(e|a|ey)?\b", "eksathe"),
           (r"\bekbare\b", "eksathe"), (r"\bat a time\b|\bat once\b|\btogether\b", "at the same time"), (r"\bkoto\s*(gula|ta|ti)\b|\bkotogula\b", "koyta"),
           (r"\b(dewa|deoa|dea)\b", "deya"), (r"\b(newa|neoa)\b", "newa")]

# ----------------------------------------------------------------------------- Bangla loanwords
# Students write English words in Bangla script ("সাপ্লি পরীক্ষার ফি"), while many dataset questions keep them in
# Latin script ("Supplementary পরীক্ষার fee"). The English word is added for BM25 and an English-loanword version of
# the question is used as an extra variant for the dense model and the reranker.
BN_LOAN = {
    "সাপ্লি": "supplementary", "সাপ্লিমেন্টারি": "supplementary", "সাপ্লাই": "supplementary", "রিটেক": "retake",
    "ফি": "fee", "পরীক্ষা": "exam", "রেজাল্ট": "result", "ফলাফল": "result", "রুটিন": "routine", "ক্লাস": "class",
    "কোর্স": "course", "ক্রেডিট": "credit", "ভর্তি": "admission", "ওয়েভার": "waiver", "স্কলারশিপ": "scholarship",
    "বৃত্তি": "scholarship", "সেমিস্টার": "semester", "টার্ম": "term", "রেজিস্ট্রেশন": "registration",
    "নিবন্ধন": "registration", "অ্যাডমিট": "admit", "এডমিট": "admit", "কার্ড": "card", "ল্যাব": "lab", "বাস": "bus",
    "ডিপার্টমেন্ট": "department", "বিভাগ": "department", "শিক্ষক": "teacher", "লাইব্রেরি": "library",
    "হোস্টেল": "hostel", "ক্যান্টিন": "canteen", "ওয়াইফাই": "wifi", "পাসওয়ার্ড": "password",
    "সার্টিফিকেট": "certificate", "ট্রান্সক্রিপ্ট": "transcript", "ফাইনাল": "final", "গ্রেড": "grade",
    "জিপিএ": "gpa", "সিজিপিএ": "cgpa", "উপস্থিতি": "attendance", "অ্যাটেনডেন্স": "attendance", "নোটিশ": "notice",
    "অফিস": "office", "ড্রপ": "drop", "কিস্তি": "installment", "ব্যাচ": "batch", "সেকশন": "section", "রুম": "room",
    "ক্যাম্পাস": "campus", "ইমেইল": "email", "জরিমানা": "fine", "ফাইন": "fine", "থিসিস": "thesis",
    "প্রজেক্ট": "project", "ক্যালেন্ডার": "calendar", "ছুটি": "holiday", "ভাইভা": "viva", "ক্লাসরুম": "classroom",
}
_BN_STEMS = sorted(BN_LOAN, key=len, reverse=True)
_BN_SUF = re.compile("(?:ের|এর|র|ে|তে|কে|টি|টা|গুলো|গুলি|য়|ও)?")


def bn_loan(t):
    """English word for a Bangla-script loanword (with or without a case ending), else None"""
    for st in _BN_STEMS:
        if t.startswith(st) and _BN_SUF.fullmatch(t[len(st):]):
            return BN_LOAN[st]
    return None


def bn_loan_variant(q):
    """'সাপ্লি পরীক্ষার ফি কত?' -> 'supplementary exam fee কত?' (None when nothing changes)"""
    out = re.sub("[\u0980-\u09ff]+", lambda m: bn_loan(m.group(0)) or m.group(0), str(q))
    return out if out != str(q) else None


# ----------------------------------------------------------------------------- teacher short names
# short names / Bangla names of CSE routine teachers -> full name (from the routine's faculty initial list).
# A question that uses only a short name ("rima mam ki class nen?") also gets the full name, so it can match
# the teacher's facts. Only names that belong to exactly one teacher are listed.
TEACHER_ALIAS = {
    'archi': 'Archi Arani Basak',
    'আর্চি': 'Archi Arani Basak',
    'rafsan': 'Ahmed Afif Rafsan',
    'afif': 'Ahmed Afif Rafsan',
    'রাফসান': 'Ahmed Afif Rafsan',
    'alaul': 'Md. Alaul Haque',
    'আলাউল': 'Md. Alaul Haque',
    'aisha': 'Aisha Hayder Chowdhury',
    'আয়েশা': 'Aisha Hayder Chowdhury',
    'istiakur': 'Ahmed Istiakur Rahman',
    'ইস্তিয়াকুর': 'Ahmed Istiakur Rahman',
    'abu jafar': 'Abu Jafar Md Jakaria',
    'আবু জাফর': 'Abu Jafar Md Jakaria',
    'abdullah al masud': 'Abdullah Al Masud',
    'আব্দুল্লাহ আল মাসুদ': 'Abdullah Al Masud',
    'ashikur': 'Ashikur Rahman',
    'আশিকুর': 'Ashikur Rahman',
    'anika': 'Anika Tabassum',
    'অনিকা': 'Anika Tabassum',
    'wadud': 'Abdul Wadud Shakib',
    'ওয়াদুদ': 'Abdul Wadud Shakib',
    'musa': 'Abu Yousuf Musa',
    'মুসা': 'Abu Yousuf Musa',
    'bushra': 'Bushra Azmat Hussain',
    'বুশরা': 'Bushra Azmat Hussain',
    'mukammel': 'Professor Choudhury M. Mukammel Wahid',
    'মুকাম্মেল': 'Professor Choudhury M. Mukammel Wahid',
    'fardin': 'Md. Fardin Ahasan Maraz',
    'maraz': 'Md. Fardin Ahasan Maraz',
    'ফারদিন': 'Md. Fardin Ahasan Maraz',
    'farhana akter': 'Farhana Akter',
    'ফারহানা আক্তার': 'Farhana Akter',
    'fahmidur': 'Md. Fahmidur Rahman Sakib',
    'fahmid': 'Md. Fahmidur Rahman Sakib',
    'ফাহমিদুর': 'Md. Fahmidur Rahman Sakib',
    'naeem': 'Golam Mostofa Naeem',
    'naim': 'Golam Mostofa Naeem',
    'নাঈম': 'Golam Mostofa Naeem',
    'mahdi': 'Md. Imam Mahdi',
    'মাহদী': 'Md. Imam Mahdi',
    'ishrar': 'Ishrar Nazah Chowdhury',
    'ইশরার': 'Ishrar Nazah Chowdhury',
    'mayami': 'Mayami Das Purkayastha Purba',
    'purba': 'Mayami Das Purkayastha Purba',
    'পূর্বা': 'Mayami Das Purkayastha Purba',
    'mahfujul': 'Md. Mahfujul Hasan',
    'mahfuj': 'Md. Mahfujul Hasan',
    'মাহফুজ': 'Md. Mahfujul Hasan',
    'muzammil': 'Muhammad Muzammil',
    'মুজাম্মিল': 'Muhammad Muzammil',
    'limon': 'Md. Shamihul Islam Khan Limon',
    'shamihul': 'Md. Shamihul Islam Khan Limon',
    'লিমন': 'Md. Shamihul Islam Khan Limon',
    'rafee': 'Md. Mushtaq Shahriyar Rafee',
    'rafi': 'Md. Mushtaq Shahriyar Rafee',
    'mushtaq': 'Md. Mushtaq Shahriyar Rafee',
    'রাফি': 'Md. Mushtaq Shahriyar Rafee',
    'shadman': 'Md. Shadman Shakib',
    'শাদমান': 'Md. Shadman Shakib',
    'nabila': 'Nabila Jannat Rifa',
    'rifa': 'Nabila Jannat Rifa',
    'নাবিলা': 'Nabila Jannat Rifa',
    'ruhul': 'Ruhul Amin',
    'রুহুল': 'Ruhul Amin',
    'raisa': 'Raisa Fairooz',
    'fairooz': 'Raisa Fairooz',
    'রাইসা': 'Raisa Fairooz',
    'sadman': 'Sadman Sakib',
    'সাদমান': 'Sadman Sakib',
    'shrabanti': 'Shrabanti Chowdhury',
    'srabonti': 'Shrabanti Chowdhury',
    'shrabonti': 'Shrabanti Chowdhury',
    'srabanti': 'Shrabanti Chowdhury',
    'শ্রাবন্তী': 'Shrabanti Chowdhury',
    'rima': 'Samia Rahman Rima',
    'samia': 'Samia Rahman Rima',
    'রিমা': 'Samia Rahman Rima',
    'tofayel': 'Professor Dr. Tofayel Ahmed',
    'তোফায়েল': 'Professor Dr. Tofayel Ahmed',
    'tajbin': 'Tajbin Jahan',
    'তাজবিন': 'Tajbin Jahan',
}
_ALIAS_LAT = sorted((a for a in TEACHER_ALIAS if a.isascii()), key=len, reverse=True)
_ALIAS_BN = [a for a in TEACHER_ALIAS if not a.isascii()]


def with_teacher_name(q):
    """append the full teacher name when only a short name is used (returns q unchanged otherwise)"""
    s = str(q); low = s.lower(); add = []
    for a in _ALIAS_LAT:
        if re.search(r"(?<![a-z])" + re.escape(a) + r"(?![a-z])", low):
            add.append(TEACHER_ALIAS[a])
    for a in _ALIAS_BN:                      # whole word only (+ possessive), so e.g. "পূর্বাভাস" does not match "পূর্বা"
        if re.search(r"(?<![\u0980-\u09ff])" + re.escape(a) + r"(?:ের|র|কে)?(?![\u0980-\u09ff])", s):
            add.append(TEACHER_ALIAS[a])
    add = [n for n in dict.fromkeys(add) if n.lower() not in low]
    return s if not add else s.rstrip(" ?।") + " (" + ", ".join(add) + ")"

# ----------------------------------------------------------------------------- refusal routing
EXAM_TOPIC = re.compile(
    r"(exam|suppl|supply|supli|sapli|suply|supplementary|retake|retek|result|admit|grade|gpa|cgpa|marks?\b|porikkha|semester drop|"
    r"পরীক্ষা|সাপ্লি|সাপ্লিমেন্টারি|রিটেক|ফলাফল|রেজাল্ট|অ্যাডমিট|এডমিট|গ্রেড|সিজিপিএ)", re.I)
MU_TOPIC = re.compile(
    r"(\bmu\b|metropolitan|university|varsity|campus|department|dept|teacher|faculty|\bsir\b|madam|fee|waiver|scholarship|"
    r"admission|vorti|bhorti|class|routine|course|credit|\bbus\b|library|hostel|office|semester|\bterm\b|cse|eee|bba|swe|llb|"
    r"মেট্রোপলিটন|ভার্সিটি|বিশ্ববিদ্যালয়|ইউনিভার্সিটি|ক্যাম্পাস|বিভাগ|ডিপার্টমেন্ট|শিক্ষক|স্যার|ফি|ভর্তি|ক্লাস|রুটিন|কোর্স|ক্রেডিট|বাস|লাইব্রেরি|অফিস|সেমিস্টার)", re.I)

REFUSE_MSG = {
    "exam": {
        "en": "I'm sorry, I don't have that information. Please contact the Exam Office (Room 108, 1st floor; 01757-537754 or +8802996642698; exam@metrouni.edu.bd).",
        "bn": "দুঃখিত, এই তথ্য আমার কাছে নেই। Exam Office-এ যোগাযোগ করো (Room 108, 1st floor; 01757-537754 বা +8802996642698; exam@metrouni.edu.bd)।",
        "banglish": "Dukkhito, ei tottho amar kache nei. Exam Office e jogajog koro (Room 108, 1st floor; 01757-537754 ba +8802996642698; exam@metrouni.edu.bd).",
    },
    "mu": {
        "en": "I'm sorry, I don't know that. Please contact the Metropolitan University One Stop service (01757-537754, 1stop@metrouni.edu.bd) or info@metrouni.edu.bd.",
        "bn": "দুঃখিত, এটা আমি জানি না। Metropolitan University-র One Stop service (01757-537754, 1stop@metrouni.edu.bd) অথবা info@metrouni.edu.bd-তে যোগাযোগ করো।",
        "banglish": "Dukkhito, eta ami jani na. Metropolitan University r One Stop service (01757-537754, 1stop@metrouni.edu.bd) othoba info@metrouni.edu.bd e jogajog koro.",
    },
    "other": {
        "en": "Sorry, I only have information about Metropolitan University, Sylhet. Please check that university's own website or office.",
        "bn": "দুঃখিত, আমার কাছে শুধু Metropolitan University, Sylhet-এর তথ্য আছে। ওই university-র নিজস্ব website বা office-এ খোঁজ নাও।",
        "banglish": "Dukkhito, amar kache shudhu Metropolitan University, Sylhet er tottho ache. Oi university r nijer website ba office e khoj nao.",
    },
    "ood": {
        "en": "Sorry, I can only answer questions about Metropolitan University, Sylhet (courses, routines, exams, fees, faculty, bus and campus).",
        "bn": "দুঃখিত, আমি শুধু Metropolitan University, Sylhet সম্পর্কিত প্রশ্নের উত্তর দিতে পারি (course, routine, পরীক্ষা, fee, faculty, bus ও campus)।",
        "banglish": "Dukkhito, ami shudhu Metropolitan University, Sylhet niye proshner uttor dite pari (course, routine, exam, fee, faculty, bus ar campus).",
    },
}


OTHER_UNI = re.compile(
    r"\b(brac|nsu|north\s*south|east\s*west|ewu|aiub|american international|iub|independent university|uiu|united international|"
    r"daffodil|diu|buet|dhaka university|university of dhaka|rajshahi university|chittagong university|jahangirnagar|"
    r"sust|shahjalal|sau|sylhet agricultural|leading university|north\s*east university|neub|sylhet international|siu|iut|"
    r"kuet|ruet|cuet|ulab|aust|ahsanullah|bup|manarat|stamford|green university|southeast university|presidency university|"
    r"harvard|oxford|cambridge|\bmit\b|stanford)\b|ব্র্যাক|নর্থ সাউথ|শাবিপ্রবি|শাহজালাল|ঢাকা বিশ্ববিদ্যালয়|বুয়েট|লিডিং ইউনিভার্সিটি|"
    r"নর্থ ইস্ট|রাজশাহী বিশ্ববিদ্যালয়|চট্টগ্রাম বিশ্ববিদ্যালয়|জাহাঙ্গীরনগর|ড্যাফোডিল|ইস্ট ওয়েস্ট", re.I)


def mentions_other_university(q):
    return bool(OTHER_UNI.search(str(q))) and not re.search(r"metropolitan|\bmu\b|মেট্রোপলিটন", str(q), re.I)


def refusal_route(q, status="ok"):
    if status == "other_university":
        return "other"
    if EXAM_TOPIC.search(str(q)):
        return "exam"
    if status == "unknown_entity" or MU_TOPIC.search(str(q)):
        return "mu"
    return "ood"


# ----------------------------------------------------------------------------- entities
BATCH_RE = re.compile(r"(?<![\d.:/])\b([5-7]\d)\s*-?\s*([a-i](?:\s*\+\s*[a-i])*)\b(?![.:]\w)", re.I)
CODE_RE = re.compile(r"\b([a-z]{2,4})\s*-?\s*(\d{3,4})(?:\s*-?\s*(\d{4}))?\b", re.I)
CODE2_RE = re.compile(r"\b([a-z]{2,4})-(\d{1,2})\b", re.I)
BUS_RE = re.compile(r"\bbus\s*(?:no\.?|number|nombor|nong|num)?\s*#?\s*(\d{1,2})\b(?![:.]\d)", re.I)
BUS_BN_RE = re.compile(r"(\d{1,2})\s*(?:নম্বর|নং)\s*বাস|বাস\s*(?:নম্বর|নং)?\s*(\d{1,2})\b(?![:.]\d)")


RT_RE = re.compile(r"(?<![A-Za-z0-9&+])[A-Z][A-Z&]{1,5}(?![A-Za-z0-9&])")


def _batches(text):
    out = set()
    for m in BATCH_RE.finditer(str(text).translate(BN_DIGITS)):
        for letter in re.split(r"\s*\+\s*", m.group(2)):
            out.add(f"{m.group(1)} {letter.upper()}")
    return out


def _codes(text, prefixes):
    t = str(text).translate(BN_DIGITS)
    out = set()
    for m in CODE_RE.finditer(t):
        p = m.group(1).upper()
        if p in prefixes:
            digits = m.group(2) + (m.group(3) or "")
            out.add((p, digits, m.group(3) or m.group(2)))
    for m in CODE2_RE.finditer(t):
        p = m.group(1).upper()
        if p in prefixes:
            out.add((p, m.group(2), m.group(2)))
    return out


def _buses(text):
    t = str(text).translate(BN_DIGITS)
    out = {str(int(m.group(1))) for m in BUS_RE.finditer(t)}
    for m in BUS_BN_RE.finditer(t):
        out.add(str(int(m.group(1) or m.group(2))))
    return out


def _year_like(code):
    p, digits, last = code
    return len(digits) == 4 and 2019 <= int(digits) <= 2031


# ----------------------------------------------------------------------------- main class
class MURAG:
    """embed(texts) -> L2-normalised np.ndarray ; rerank(query, passages) -> np.ndarray of scores in [0, 1]."""

    W = {"dense_q": 1.0, "dense_a": 0.5, "bm25_q": 0.7, "bm25_a": 0.3, "char_q": 0.4}

    def __init__(self, kb, facts, embed=None, rerank=None, n_rerank=20, rrf_k=60):
        self.kb = kb.reset_index(drop=True).copy()
        self.facts = facts.set_index("fact_id", drop=False)
        self.fids = list(self.facts.index)
        self.fpos = {f: i for i, f in enumerate(self.fids)}
        self.nf = len(self.fids)
        self.row_f = np.array([self.fpos[f] for f in self.kb["fact_id"]])
        self.kind = self.facts["kind"].to_dict()
        self.embed, self.rerank_fn, self.n_rerank, self.rrf_k = embed, rerank, n_rerank, rrf_k
        self.ans_docs = [(self.fpos[f], lang, str(self.facts.at[f, col])) for f in self.fids for lang, col in LANG_COL.items()]
        self.ans_f = np.array([d[0] for d in self.ans_docs])
        self._q_cache, self._rr_cache, self._tok_cache, self._ch_cache = {}, {}, {}, {}
        self._build_lexical()
        self._build_entities()
        self._build_language_model()
        self._build_rare_tokens()
        if embed is not None:
            self.q_emb = np.asarray(embed(list(self.kb["question"])), dtype="float32")
            self.a_emb = np.asarray(embed([d[2][:1200] for d in self.ans_docs]), dtype="float32")

    # ------------------------------------------------------------------ building
    def _build_lexical(self):
        qs = list(self.kb["question"])
        self.vocab = Counter(t for q in qs for t in tokens(q))
        self.vocab_lat = [w for w in self.vocab if w.isascii() and w.isalpha() and len(w) >= 3]
        self.vocab_bn = [w for w in self.vocab if not w.isascii() and len(w) >= 3]
        self.bm25_q = BM25Okapi([tokens(q) for q in qs])
        self.bm25_a = BM25Okapi([tokens(d[2]) for d in self.ans_docs])
        self.char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit(qs)
        self.char_q = self.char.transform(qs)
        # representative question of each fact per language (first dataset row) for the reranker passage
        self.rep_q = defaultdict(dict)
        for r in self.kb.itertuples():
            self.rep_q[self.fpos[r.fact_id]].setdefault(r.language, r.question)

    def _build_entities(self):
        cats = self.facts["category"].to_dict()
        code_rows = self.kb[self.kb["fact_id"].map(cats).isin(["curriculum", "supplementary routine"])]
        self.prefixes = {m.group(1).upper() for q in code_rows["question"] for m in CODE_RE.finditer(q)}
        self.prefixes |= {m.group(1).upper() for q in code_rows["question"] for m in CODE2_RE.finditer(q)}
        self.fact_ent = defaultdict(lambda: {"batch": set(), "code": set(), "bus": set()})
        for r in self.kb.itertuples():
            fe = self.fact_ent[self.fpos[r.fact_id]]
            fe["batch"] |= _batches(r.question)
            fe["code"] |= _codes(r.question, self.prefixes)
            fe["bus"] |= _buses(r.question)
        self.ans_ent = {"batch": set(), "code": set(), "bus": set()}
        for _, _, a in self.ans_docs:
            self.ans_ent["batch"] |= _batches(a)
            self.ans_ent["code"] |= _codes(a, self.prefixes)
        self.ans_ent["bus"] = {b for fe in self.fact_ent.values() for b in fe["bus"]}
        # routine abbreviations / faculty initials ("MP&I", "DBMS", "SAS"): case-sensitive, soft filter only
        rt_rows = self.kb[self.kb["fact_id"].map(cats) == "class routine"]
        other = Counter(t for q in self.kb[self.kb["fact_id"].map(cats) != "class routine"]["question"] for t in set(RT_RE.findall(q)))
        cand = Counter(t for q in rt_rows["question"] for t in set(RT_RE.findall(q)))
        self.rt_vocab = {t for t in cand if not other[t] and t not in self.prefixes and (len(t) >= 3 or "&" in t)
                         and t not in {"CSE", "AUTUMN"}}
        self.rt_facts = defaultdict(set)
        for r in rt_rows.itertuples():
            for t in set(RT_RE.findall(r.question)) & self.rt_vocab:
                self.rt_facts[t].add(self.fpos[r.fact_id])

    def _build_language_model(self):
        cnt = {"en": Counter(), "banglish": Counter()}
        for r in self.kb.itertuples():
            if r.language in cnt:
                cnt[r.language].update(t for t in tokens(r.question) if t.isalpha())
        tot = {k: sum(v.values()) for k, v in cnt.items()}
        V = len(set(cnt["en"]) | set(cnt["banglish"])) + 1
        self.lang_llr = {}
        for w in set(cnt["en"]) | set(cnt["banglish"]):
            pb = (cnt["banglish"][w] + 1) / (tot["banglish"] + V)
            pe = (cnt["en"][w] + 1) / (tot["en"] + V)
            self.lang_llr[w] = math.log(pb / pe)

    def _build_rare_tokens(self):
        self.df = {}
        for lang, col in LANG_COL.items():
            c = Counter()
            for a in self.facts[col]:
                c.update(set(self._content_tokens(a)))
            self.df[lang] = c
        self.rare_cut = max(3, int(0.01 * self.nf))

    @staticmethod
    def _content_tokens(text):
        return [t for t in tokens(text) if not t.isdigit() and (len(t) >= 3 if t.isascii() else len(t) >= 2)]

    # ------------------------------------------------------------------ language
    def detect_language(self, q):
        if BN_RE.search(str(q)):
            return "bn"
        s = sum(self.lang_llr.get(t, 0.0) for t in tokens(q) if t.isalpha())
        return "banglish" if s > 0.5 else "en"

    # ------------------------------------------------------------------ normalisation
    def _correct(self, t):
        if t in self._tok_cache:
            return self._tok_cache[t]
        out = [t]
        if t not in self.vocab and len(t) >= 4 and not t.isdigit():
            pool, cut = (self.vocab_lat, 82) if t.isascii() else (self.vocab_bn, 85)
            m = process.extractOne(t, pool, scorer=fuzz.ratio, score_cutoff=cut)
            if m and abs(len(m[0]) - len(t)) <= 2:
                out.append(m[0])
        self._tok_cache[t] = out
        return out

    def normalize(self, q):
        q = str(q).lower().translate(BN_DIGITS)
        for pat, rep in PHRASES:
            q = re.sub(pat, rep, q)
        out = []
        for t in tokens(q):
            t = CANON.get(t, t)
            if not t:
                continue
            for c in self._correct(t):
                out.append(c)
                out.extend(EXPAND.get(c, []))
            if not t.isascii():
                e = bn_loan(t)
                if e:
                    out.append(e); out.extend(EXPAND.get(e, []))
        return out

    def clean_query(self, q):
        """spelling-corrected version of the query (same words, typos fixed against the dataset vocabulary)"""
        t = str(q).lower().translate(BN_DIGITS)
        for pat, rep_ in PHRASES:
            t = re.sub(pat, rep_, t)
        out = []
        for w in tokens(t):
            w = CANON.get(w, w)
            if not w:
                continue
            c = self._correct(w)
            out.append(c[1] if len(c) > 1 else w)
        return " ".join(out)

    def variants(self, q, lang=None):
        """the raw query + its spelling-corrected form (+ an English gloss for Banglish); dense and reranker
        scores are the maximum over these variants, which helps Banglish and misspelled questions"""
        lang = lang or self.detect_language(q)
        v = [str(q).strip()]
        c = self.clean_query(q)
        if c and c != v[0].lower():
            v.append(c)
        if lang == "banglish":
            g = " ".join(x for x in (GLOSS.get(w, w) for w in (c or str(q).lower()).split()) if x)
            if g and g not in v:
                v.append(g)
        if lang == "bn":
            b = bn_loan_variant(q)                  # "সাপ্লি পরীক্ষার ফি কত?" -> "supplementary exam fee কত?"
            if b:
                v.append(b)
        t = with_teacher_name(q)                    # "rima mam ki class nen?" -> "... (Samia Rahman Rima)"
        if t != str(q):
            v.append(t)
        return list(dict.fromkeys(v))

    # ------------------------------------------------------------------ entity filter
    def entity_filter(self, q):
        """returns (status, allowed fact positions or None, detail, hard) - hard = an exact batch/code/bus matched"""
        q_ent = {"batch": _batches(q), "code": {c for c in _codes(q, self.prefixes)}, "bus": _buses(q)}
        allowed = None
        for typ, ents in q_ent.items():
            for e in ents:
                if typ == "code":
                    match = {f for f, fe in self.fact_ent.items()
                             if any(c[0] == e[0] and e[1] in (c[1], c[2]) for c in fe["code"])}
                    in_answers = any(c[0] == e[0] and e[1] in (c[1], c[2]) for c in self.ans_ent["code"])
                    if not match and not in_answers and _year_like(e):
                        continue
                else:
                    match = {f for f, fe in self.fact_ent.items() if e in fe[typ]}
                    in_answers = e in self.ans_ent[typ]
                if match:
                    allowed = match if allowed is None else (allowed & match or allowed | match)
                elif not in_answers:
                    return "unknown_entity", None, f"{typ}:{e if typ != 'code' else e[0] + ' ' + e[1]}", False
        hard = allowed is not None
        if allowed is None:                     # soft: only narrows, never refuses
            rts = [self.rt_facts[t] for t in set(RT_RE.findall(str(q))) if t in self.rt_vocab]
            if rts:
                allowed = set.intersection(*rts) or set.union(*rts)
        return "ok", allowed, "", hard

    # ------------------------------------------------------------------ scoring channels (one score per fact)
    def _fact_max(self, row_scores, owner):
        out = np.full(self.nf, -1e9)
        np.maximum.at(out, owner, row_scores)
        return out

    def _qvec(self, q):
        if q not in self._q_cache:
            self._q_cache[q] = np.asarray(self.embed([q])[0], dtype="float32")
        return self._q_cache[q]

    def channels(self, q):
        if q in self._ch_cache:
            return self._ch_cache[q]
        ntoks = self.normalize(q)
        nq = " ".join(ntoks) or str(q).lower()
        ch = {}
        if self.embed is not None:
            v = self._qvec(q)
            ch["dense_q"] = self._fact_max(self.q_emb @ v, self.row_f)
            ch["dense_a"] = self._fact_max(self.a_emb @ v, self.ans_f)
            raw_q, raw_a = ch["dense_q"], ch["dense_a"]
            for extra in self.variants(q)[1:]:
                ve = self._qvec(extra)
                raw_q = np.maximum(raw_q, self._fact_max(self.q_emb @ ve, self.row_f))
                raw_a = np.maximum(raw_a, self._fact_max(self.a_emb @ ve, self.ans_f))
            ch["dense_q_var"], ch["dense_a_var"] = raw_q, raw_a
        ch["bm25_q"] = self._fact_max(self.bm25_q.get_scores(ntoks), self.row_f)
        ch["bm25_a"] = self._fact_max(self.bm25_a.get_scores(ntoks), self.ans_f)
        ch["char_q"] = self._fact_max((self.char_q @ self.char.transform([nq]).T).toarray().ravel(), self.row_f)
        if len(self._ch_cache) < 20000:
            self._ch_cache[q] = ch
        return ch

    def rrf(self, ch, use=None, allowed=None):
        """weighted Reciprocal Rank Fusion. With an entity filter the ranks are computed among the allowed facts only,
        so a fact that exactly matches the batch / code is not penalised by facts that were filtered out."""
        total = np.zeros(self.nf)
        idx = np.arange(self.nf) if allowed is None else np.fromiter(sorted(allowed), dtype=int)
        for name, s in ch.items():
            if name not in self.W or (use is not None and name not in use):
                continue
            if name in ("dense_q", "dense_a") and name + "_var" in ch:
                s = ch[name + "_var"]          # hybrid systems use the best of the query variants
            sub = s[idx]
            pos = np.empty(len(idx))
            pos[np.argsort(-sub, kind="stable")] = np.arange(len(idx))
            total[idx] += self.W[name] / (self.rrf_k + pos + 1)
        return total

    # ------------------------------------------------------------------ rankers
    def _ranking(self, scores, allowed=None, k=10):
        if allowed is not None:
            mask = np.full(self.nf, -np.inf)
            idx = np.fromiter(allowed, dtype=int)
            mask[idx] = scores[idx]
            scores = mask
        order = np.argsort(-scores, kind="stable")[:k]
        return [(self.fids[i], float(scores[i])) for i in order if np.isfinite(scores[i])]

    def passage(self, f, lang):
        qrep = self.rep_q[f].get(lang) or self.rep_q[f].get("en")
        acol = "answer_bn" if lang == "bn" else "answer_en"
        return f"{qrep}\n{str(self.facts.iat[f, self.facts.columns.get_loc(acol)])[:700]}"

    def rerank(self, q, cand_pos, lang):
        key = (q, tuple(cand_pos))
        if key not in self._rr_cache:
            passages = [self.passage(f, lang) for f in cand_pos]
            s = None
            for v in self.variants(q, lang):        # best score over raw / corrected / glossed query
                sv = np.asarray(self.rerank_fn(v, passages), dtype="float64")
                s = sv if s is None else np.maximum(s, sv)
            self._rr_cache[key] = s
        return self._rr_cache[key]

    def retrieve(self, q, system="final", k=10):
        """system: bm25 | dense | hybrid | hybrid_entity | final
        returns dict(ranking=[(fact_id, score)], status, conf, margin, sim, lang)
          conf   = the system's own score of the top fact (cosine / fused RRF / reranker probability)
          margin = conf(top-1) - conf(top-2)
          sim    = absolute evidence for the top fact: its dense cosine (char-n-gram cosine without a dense model),
                   1.0 when an exact batch / course code / bus number matched. RRF scores are rank-based, so a
                   refusal rule needs this absolute signal as well."""
        lang = self.detect_language(q)
        status, allowed, detail, hard = "ok", None, "", False
        if system in ("hybrid_entity", "final"):
            if mentions_other_university(q):        # never answer MU facts for a question about another university
                return dict(ranking=[], status="other_university", detail="", conf=0.0, margin=0.0, sim=0.0, lang=lang)
            status, allowed, detail, hard = self.entity_filter(q)
            if status == "unknown_entity":
                return dict(ranking=[], status=status, detail=detail, conf=0.0, margin=0.0, sim=0.0, lang=lang)
        ch = self.channels(q)
        if system == "dense":                      # old RAG baseline: raw query only
            absolute = ch["dense_q"] if "dense_q" in ch else ch["char_q"]
        else:
            absolute = ch.get("dense_q_var", ch.get("dense_q", ch["char_q"]))
        if system == "bm25":
            scores = ch["bm25_q"]
        elif system == "dense":
            scores = absolute
        else:
            scores = self.rrf(ch, allowed=allowed)
        hybrid_top = self._ranking(scores, allowed, k=1)
        consensus = False
        if system == "final" and self.rerank_fn is not None:
            # rerank the union of the hybrid top-N and the dense top-N (inside the entity filter)
            cand = [self.fpos[f] for f, _ in self._ranking(scores, allowed, k=self.n_rerank)]
            cand += [self.fpos[f] for f, _ in self._ranking(absolute, allowed, k=self.n_rerank // 2)]
            cand = list(dict.fromkeys(cand))
            rr = self.rerank(q, cand, lang)
            order = np.argsort(-rr, kind="stable")
            ranking = [(self.fids[cand[i]], float(rr[i])) for i in order]
            # the cross-encoder is weak on code-mixed Banglish: when it is unsure (top score < 0.5) but the dense
            # retriever and the hybrid (lexical + dense) retriever independently rank the SAME fact first, that fact
            # is put first. It can then only be answered through the "retrievers agree and similarity >= alpha"
            # path of the rule, whose alpha is tuned on dev with the wrong-answer penalty.
            dense_top = self._ranking(absolute, allowed, k=1)
            if (ranking and ranking[0][1] < CONSENSUS_MAX_CONF and hybrid_top and dense_top
                    and hybrid_top[0][0] == dense_top[0][0] != ranking[0][0]):
                pick = [x for x in ranking if x[0] == hybrid_top[0][0]]
                if pick:
                    ranking = pick + [x for x in ranking if x[0] != pick[0][0]]
                    consensus = True
        else:
            ranking = self._ranking(scores, allowed, k=max(k, 2))
        conf = ranking[0][1] if ranking else 0.0
        margin = conf - ranking[1][1] if len(ranking) > 1 else conf
        dense = float(absolute[self.fpos[ranking[0][0]]]) if ranking else 0.0
        sim = 1.0 if hard else dense
        agree = bool(ranking) and bool(hybrid_top) and hybrid_top[0][0] == ranking[0][0]
        return dict(ranking=ranking[:k], status=status, detail=detail, conf=conf, margin=margin, sim=sim, dense=dense,
                    agree=agree, lang=lang, consensus=consensus)

    # ------------------------------------------------------------------ decision
    @staticmethod
    def decide(r, rule):
        """Answer (True) or say "I don't know" (False). rule = {"direct": cos or None, "by_lang": {lang: sub-rule}}
        with sub-rule = {"tau": reranker score, "sigma": similarity floor, "alpha": floor when retrievers agree}.
        Answer if: the question is almost identical to a dataset question (dense >= direct), OR
                   score >= tau and similarity >= sigma, OR
                   the reranker and the hybrid retriever pick the same fact and similarity >= alpha."""
        if r["status"] != "ok" or not r["ranking"]:
            return False
        if rule.get("direct") is not None and r.get("dense", 0.0) >= rule["direct"]:
            return True
        R = rule.get("by_lang", {}).get(r.get("lang"), rule)
        if r["conf"] >= R["tau"] and r.get("sim", 1.0) >= R.get("sigma", -1.0):
            return True
        a = R.get("alpha")
        return a is not None and bool(r.get("agree")) and r.get("sim", 0.0) >= a

    @staticmethod
    def accept(r, tau, mu=0.0, sigma=-1.0, direct=None):
        """answer if (score >= tau and similarity >= sigma), or if the question is almost identical to a dataset
        question (dense cosine >= direct). Unknown entities / other universities are never answered."""
        if r["status"] != "ok" or not r["ranking"]:
            return False
        if direct is not None and r.get("dense", 0.0) >= direct:
            return True
        return r["conf"] >= tau and r["margin"] >= mu and r.get("sim", 1.0) >= sigma

    def verified_answer(self, fid, lang):
        return str(self.facts.at[fid, LANG_COL[lang]])

    # ------------------------------------------------------------------ grounding guard / key-fact coverage
    def key_items(self, ref, lang):
        rare = []
        df = self.df[lang if lang in self.df else "en"]
        seen = set()
        for t in self._content_tokens(ref):
            if t in seen:
                continue
            seen.add(t)
            d = df.get(t, 0)
            if d <= self.rare_cut:
                rare.append((d, -len(t), t))
        rare = [t for _, _, t in sorted(rare)[:8]]
        return dict(numbers=numbers(ref), emails=emails(ref), rare=rare, polarity=polarity(ref))

    def covers(self, answer, ref, lang):
        """True if `answer` states the key facts of `ref`: every number and email, at least half of the rare
        content words, and the same yes/no polarity."""
        k = self.key_items(ref, lang)
        a_low = str(answer).lower()
        a_nums, a_mail = numbers(answer), emails(answer)
        if not k["numbers"] <= a_nums or not k["emails"] <= a_mail:
            return False
        if k["rare"]:
            hit = sum(t in a_low for t in k["rare"])
            if hit < math.ceil(0.5 * len(k["rare"])):
                return False
        if k["polarity"] is not None:
            neg = bool(NEG_RE.search(str(answer)[:160]))
            if (k["polarity"] == "yes") == neg and not YES_START.search(str(answer)):
                return False
            if k["polarity"] == "no" and not neg:
                return False
        return True

    def guard(self, gen, verified, lang):
        probs = []
        g = str(gen or "").strip()
        if len(g) < 3:
            return ["empty"]
        if not script_ok(g, lang, verified):
            probs.append("wrong_language")
        extra = numbers(g) - numbers(verified)
        if extra:
            probs.append("unsupported_numbers:" + ",".join(sorted(extra)[:5]))
        if emails(g) - emails(verified):
            probs.append("unsupported_email")
        if looks_like_refusal(g) and not looks_like_refusal(verified):
            probs.append("needless_refusal")
        if not self.covers(g, verified, lang):
            probs.append("missing_key_facts")
        return probs

In [ ]:
import mu_rag; importlib.reload(mu_rag)
from mu_rag import MURAG, REFUSE_MSG, refusal_route, numbers, looks_like_refusal, script_ok

## 4. Models for retrieval

In [ ]:
if MOCK:   # tiny stand-ins so the logic can be unit-tested without GPU / internet
    from sklearn.feature_extraction.text import TfidfVectorizer
    from rapidfuzz import fuzz
    _tf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit(
        list(kb.question) + list(facts.answer_en) + list(facts.answer_bn))
    def embed(texts):
        m = _tf.transform(list(texts)).toarray()
        return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-9)
    def rerank_fn(q, passages):
        return np.array([fuzz.token_set_ratio(q.lower(), p.split("\n")[0].lower()) / 100 for p in passages])
else:
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    embedder = SentenceTransformer(EMBED_MODEL, device=DEV_RET); embedder.half()
    def embed(texts):
        return embedder.encode(list(texts), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    rr_tok = AutoTokenizer.from_pretrained(RERANK_MODEL)
    rr_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL, torch_dtype=torch.float16).to(DEV_RET).eval()
    @torch.no_grad()
    def rerank_fn(q, passages, bs=32):
        out = []
        for i in range(0, len(passages), bs):
            enc = rr_tok([[q, p] for p in passages[i:i + bs]], padding=True, truncation=True, max_length=512,
                         return_tensors="pt").to(DEV_RET)
            out.append(torch.sigmoid(rr_model(**enc).logits.view(-1).float()).cpu().numpy())
        return np.concatenate(out)

t0 = time.time()
rag = MURAG(kb, facts, embed=embed, rerank=rerank_fn, n_rerank=N_RERANK)
print(f"Index built in {time.time() - t0:.0f}s | course-code prefixes: {len(rag.prefixes)} | facts: {rag.nf}")

det = ev["question"].map(rag.detect_language)
print("Language detection accuracy on the eval questions:", round((det == ev["language"]).mean(), 3))
display(pd.crosstab(ev["language"], det, rownames=["true"], colnames=["detected"]))
for q in ["CSE 58 A er sunday class ki?", "When is the CSE-221 supply exam?", "What is the routine of CSE 70 A?", "bus 9 er driver ke?"]:
    print(q, "->", rag.entity_filter(q)[0], rag.entity_filter(q)[2])

## 5. Retrieval ablation on the test split (which fact is ranked first?)
Recall@k counts a question as solved when a gold fact is in the top k. Out-of-domain questions are excluded here and evaluated in Sections 6 and 9.

In [ ]:
SYSTEMS = {"A. BM25 only": "bm25", "B. Dense bge-m3 (old RAG)": "dense", "C. Hybrid (dense+BM25+char, RRF)": "hybrid",
           "D. Hybrid + entity filter": "hybrid_entity", "E. D + reranker (final)": "final"}

def rank_of(gold, ranking):
    return next((i for i, (f, _) in enumerate(ranking, 1) if f in gold), None)

def retrieval_runs(frame):
    rows = []
    for r in frame.itertuples():
        for name, sysname in SYSTEMS.items():
            out = rag.retrieve(r.question, sysname)
            rows.append(dict(system=name, eval_id=r.eval_id, language=r.language, type=r.type, expected=r.expected,
                             gold=r.gold, rank=rank_of(r.gold, out["ranking"]) if r.gold else None,
                             top=out["ranking"][0][0] if out["ranking"] else None,
                             conf=out["conf"], margin=out["margin"], sim=out["sim"], dense=out.get("dense", 0.0), agree=out.get("agree", False),
                             det_lang=out["lang"], status=out["status"]))
    return pd.DataFrame(rows)

def recall_table(runs):
    d = runs[runs.expected == "answer"].copy()
    d["r1"], d["r3"] = d["rank"].fillna(99) <= 1, d["rank"].fillna(99) <= 3
    d["rr"] = d["rank"].map(lambda x: 1 / x if pd.notna(x) and x else 0)
    t = d.groupby("system", sort=False).agg(**{"R@1": ("r1", "mean"), "R@3": ("r3", "mean"), "MRR": ("rr", "mean")})
    for lang in ["en", "bn", "banglish"]:
        t[f"R@1 {lang}"] = d[d.language == lang].groupby("system", sort=False)["r1"].mean()
    hard = d[d.type.str.startswith("hard")]
    t["R@1 hard near-duplicates"] = hard.groupby("system", sort=False)["r1"].mean()
    return t.round(3)

t0 = time.time()
dev_runs, test_runs = retrieval_runs(dev), retrieval_runs(test)
print(f"({time.time() - t0:.0f}s)")
ablation = recall_table(test_runs)
display(ablation)
ablation.to_csv("outputs/retrieval_ablation_test.csv")

## 6. Refusal rule tuned on the dev split
A question is answered when (a) it is almost identical to a dataset question (bge-m3 cosine ≥ 0.90), **or** (b) the reranker score is at least **τ** *and* the similarity of the chosen fact is at least **σ**, **or** (c) the reranker and the hybrid retriever independently pick the same fact *and* the similarity is at least **α**. (When the reranker is unsure — score below 0.5 — but the dense retriever and the hybrid retriever both rank the same fact first, that fact is moved to the top; it can then only be answered through path (c).) The similarity is the best bge-m3 cosine over the raw, spelling-corrected and (for Banglish) English-glossed query, or 1.0 when an exact batch, course code or bus number matched. **τ, σ and α are tuned separately for English, Bangla and Banglish**, because Banglish questions get lower scores from the pre-trained models. The values are chosen on dev to maximise `correct − WRONG_PENALTY × (wrong answers + answered out-of-domain questions)`, so the bot prefers "I don't know" over a wrong answer. The test split is never used for tuning.

In [ ]:
KIND = FACT["kind"].to_dict()

def outcome(expected, gold, top, accepted):
    gold_kind = KIND[gold[0]] if gold else None
    declined = (not accepted) or KIND.get(top) == "unknown_refer"
    if expected == "refuse":
        return "correct_refusal" if declined else "answered_ood"
    if gold_kind == "unknown_refer":                      # the right reply is itself "I don't know -> office"
        return "correct" if declined or top in gold else "wrong"
    if not accepted:
        return "abstain"
    return "correct" if top in gold else "wrong"

def accepted_mask(d, rule):
    # exactly the same decision function as the chatbot uses (MURAG.decide)
    return np.array([MURAG.decide(dict(status=st, ranking=[t] if isinstance(t, str) else [], conf=c, sim=sm, dense=de,
                                       agree=ag, lang=lg), rule)
                     for st, t, c, sm, de, ag, lg in zip(d["status"], d["top"], d["conf"], d["sim"], d["dense"],
                                                         d["agree"], d["det_lang"])], dtype=bool)

def decide(d, rule):
    return [outcome(e, g, t, a) for e, g, t, a in zip(d["expected"], d["gold"], d["top"], accepted_mask(d, rule))]

GAIN = {"correct": 1.0, "correct_refusal": 1.0, "abstain": 0.0, "wrong": -WRONG_PENALTY, "answered_ood": -WRONG_PENALTY}

def _tune_group(d, use_agree, direct):
    # grid search for one group of dev questions: tau (score) x sigma (similarity floor) x alpha (agreement floor)
    u_acc = np.array([GAIN[outcome(e, g, t, True)] if isinstance(t, str) else GAIN[outcome(e, g, t, False)]
                      for e, g, t in zip(d["expected"], d["gold"], d["top"])])
    u_dec = np.array([GAIN[outcome(e, g, t, False)] for e, g, t in zip(d["expected"], d["gold"], d["top"])])
    ok = ((d["status"] == "ok") & d["top"].map(lambda t: isinstance(t, str))).values
    conf, sim, dense = d["conf"].values, d["sim"].values, d["dense"].values
    agree = d["agree"].values.astype(bool)
    dmask = ok & (dense >= direct) if direct is not None else np.zeros(len(d), bool)
    taus = list(np.unique(np.quantile(conf, np.linspace(0, 1, 51)))) + [1e9]          # 1e9 = path switched off
    sigmas = list(np.unique([-1.0] + list(np.quantile(sim, [0.02, 0.05, 0.1, 0.15, 0.2, 0.3]))))
    alphas = (list(np.unique(np.quantile(sim, [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]))) if use_agree else []) + [None]
    grid = []
    for ti, tau in enumerate(taus):
        for si, sg in enumerate(sigmas):
            base = dmask | (ok & (conf >= tau) & (sim >= sg))
            for ai, al in enumerate(alphas):
                acc = base | (ok & agree & (sim >= al)) if al is not None else base
                u = float(np.where(acc, u_acc, u_dec).sum())
                n_wrong = int((acc & (u_acc < 0)).sum())
                grid.append((u, -n_wrong, ti, si, ai))
    top = max((g[0], g[1]) for g in grid)
    ties = sorted([g for g in grid if (g[0], g[1]) == top], key=lambda g: (g[2] + g[3] + g[4], g[2], g[3], g[4]))
    # many settings tie on dev: take one on the LOOSE side of the tied region (25 %), so the bot answers what it
    # knows, without sitting exactly on the loosest edge that dev cannot check
    _, _, ti, si, ai = ties[int(0.25 * (len(ties) - 1))]
    al = alphas[ai]
    return dict(tau=float(taus[ti]), mu=0.0, sigma=float(sigmas[si]), alpha=None if al is None else float(al)), top[0]

def tune(runs, name):
    # One rule per detected language (Banglish questions get lower similarity scores than English ones).
    # No margin rule: near-duplicate facts with the same meaning have close scores.
    d = runs[runs.system == name].reset_index(drop=True)
    is_base = name.startswith("B.")
    direct = None if is_base else DIRECT_SIM
    if is_base:                                   # old RAG baseline: one global threshold, as in the previous notebook
        sub, u = _tune_group(d, use_agree=False, direct=None)
        return dict(direct=None, by_lang={l: sub for l in ["en", "bn", "banglish"]}, dev_utility=u)
    by_lang, total = {}, 0.0
    glob_rule, _ = _tune_group(d, use_agree=True, direct=direct)
    for lang in ["en", "bn", "banglish"]:
        g = d[d["det_lang"] == lang]
        if (g["expected"] == "answer").sum() < 15:          # too few dev questions -> use the global rule
            by_lang[lang] = glob_rule; continue
        by_lang[lang], u = _tune_group(g.reset_index(drop=True), use_agree=True, direct=direct)
        total += u
    return dict(direct=direct, by_lang=by_lang, dev_utility=total)

def describe(rule):
    parts = [f"direct if cosine >= {rule['direct']}" if rule.get("direct") is not None else ""]
    for l, r in rule["by_lang"].items():
        a = f", or retrievers agree and similarity >= {r['alpha']:.3f}" if r.get("alpha") is not None else ""
        t = "off" if r["tau"] >= 1e8 else f"{r['tau']:.3f}"
        parts.append(f"[{l}] score >= {t} and similarity >= {r['sigma']:.3f}{a}")
    return " | ".join(p for p in parts if p)

def decision_summary(runs, name, rule):
    d = runs[runs.system == name].reset_index(drop=True)
    d["outcome"] = decide(d, rule)
    ind, ood = d[d.expected == "answer"], d[d.expected == "refuse"]
    c, o = Counter(ind.outcome), Counter(ood.outcome)
    return pd.Series({"Answer accuracy": c["correct"] / len(ind), "Wrong answers": c["wrong"] / len(ind),
                      "Abstained (said I don't know)": c["abstain"] / len(ind),
                      "Precision when answering": c["correct"] / max(1, c["correct"] + c["wrong"]),
                      "OOD correctly refused": o["correct_refusal"] / max(1, len(ood))}), d

RULES = {name: tune(dev_runs, name) for name in ["B. Dense bge-m3 (old RAG)", "D. Hybrid + entity filter", "E. D + reranker (final)"]}
for k, v in RULES.items():
    print(f"{k} (dev utility {v['dev_utility']:.1f}):\n   {describe(v)}")
# the final RAG retriever is chosen on DEV only: with or without the reranker, whichever is safer and more accurate
FINAL = "E. D + reranker (final)"          # hybrid + entity filter + cross-encoder reranker
FINAL_SYSTEM = SYSTEMS[FINAL]
print("Final RAG retriever:", FINAL)
_d = dev_runs[(dev_runs.system == FINAL) & (dev_runs.expected == "answer")]
print("Dev in-domain reranker score: median %.3f, 10th percentile %.3f" % (_d.conf.median(), _d.conf.quantile(0.1)))
dec_dev = pd.DataFrame({k: decision_summary(dev_runs, k, v)[0] for k, v in RULES.items()}).round(3)
dec_test = pd.DataFrame({k: decision_summary(test_runs, k, v)[0] for k, v in RULES.items()}).round(3)
print("=== DEV (used for tuning) ==="); display(dec_dev)
print("=== TEST (verified answers, no LLM) ==="); display(dec_test)
dec_test.to_csv("outputs/decision_test_retrieval_only.csv")

## 7. Typo robustness
The same test questions get realistic spelling mistakes: dropped Bangla vowel signs, and swapped, deleted or doubled letters in Latin script. The rules stay the ones tuned on clean dev questions.

In [ ]:
BN_SIGNS = "ািীুূৃেৈোৌ্ঁংঃ"
def add_typos(q, rng):
    words = q.split()
    idx = [i for i, w in enumerate(words) if len(re.sub(r"\W", "", w)) >= 4 and not re.search(r"\d", w)]
    if not idx: return q
    for i in rng.sample(idx, max(1, round(0.35 * len(idx)))):
        w = words[i]
        if re.search(r"[ঀ-৿]", w):
            signs = [j for j, ch in enumerate(w) if ch in BN_SIGNS]
            if signs:
                j = rng.choice(signs); w = w[:j] + w[j + 1:]
        else:
            j = rng.randrange(1, len(w) - 1); op = rng.choice(["del", "swap", "dup", "sub"])
            if op == "del":  w = w[:j] + w[j + 1:]
            if op == "swap": w = w[:j] + w[j + 1] + w[j] + w[j + 2:]
            if op == "dup":  w = w[:j] + w[j] + w[j:]
            if op == "sub":  w = w[:j] + rng.choice("aeiou") + w[j + 1:]
        words[i] = w
    return " ".join(words)

rng = random.Random(SEED)
noisy = test[test.expected == "answer"].copy()
noisy["question"] = [add_typos(q, rng) for q in noisy["question"]]
display(pd.DataFrame({"clean": test[test.expected == "answer"]["question"].head(6).values, "with typos": noisy["question"].head(6).values}))
noisy_runs = retrieval_runs(noisy)
typo = pd.DataFrame({"R@1 clean": recall_table(test_runs)["R@1"], "R@1 typos": recall_table(noisy_runs)["R@1"]})
typo["drop"] = (typo["R@1 clean"] - typo["R@1 typos"]).round(3)
for k, v in RULES.items():
    if k == "D. Hybrid + entity filter" and FINAL != k: continue
    typo.loc[k, "Accuracy with typos (tuned rule)"] = decision_summary(noisy_runs, k, v)[0]["Answer accuracy"]
    typo.loc[k, "Wrong answers with typos"] = decision_summary(noisy_runs, k, v)[0]["Wrong answers"]
display(typo.round(3)); typo.round(3).to_csv("outputs/typo_robustness.csv")

## 8. The LLM and the two answer pipelines (LLM-only vs RAG)

In [ ]:
if MOCK:
    class _MockLLM: n = 0
    llm, tok = _MockLLM(), None
    def chat_generate_batch(model, tok, msgs_list, max_new_tokens=MAX_NEW_TOKENS):
        outs = []
        for m in msgs_list:
            model.n += 1
            u = m[-1]["content"]
            if "Verified answer:" in u:
                v = u.split("Verified answer:\n", 1)[1].split("\n\nStudent question:", 1)[0]
                outs.append(re.sub(r"\d+", "999", v, count=1) if model.n % 4 == 0 else v)   # corrupt some -> guard test
            else:
                outs.append("I don't know." if model.n % 2 else "The fee is 5000 taka and the office is in Room 12.")
        return outs
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    try:
        tok = AutoTokenizer.from_pretrained(LLM_MODEL)
        llm = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL, device_map={"": DEV_LLM},
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=torch.float16)).eval()
    except Exception as e:      # e.g. bitsandbytes problem -> smaller model of the same family in fp16 (no quantisation)
        print("4-bit loading failed:", type(e).__name__, str(e)[:200])
        LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
        print("Falling back to", LLM_MODEL, "(fp16)")
        gc.collect(); torch.cuda.empty_cache()
        tok = AutoTokenizer.from_pretrained(LLM_MODEL)
        llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map={"": DEV_LLM}).eval()
    llm.generation_config.temperature = None; llm.generation_config.top_p = None; llm.generation_config.top_k = None
    print("LLM loaded:", LLM_MODEL)

    @torch.no_grad()
    def _generate(model, tok, msgs_list, max_new_tokens):
        tok.padding_side = "left"
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        n = enc["input_ids"].shape[1]
        return [tok.decode(o[n:], skip_special_tokens=True).strip() for o in out]

    def chat_generate_batch(model, tok, msgs_list, max_new_tokens=MAX_NEW_TOKENS):
        try:
            return _generate(model, tok, msgs_list, max_new_tokens)
        except torch.cuda.OutOfMemoryError:          # too big a batch -> answer one by one instead of crashing
            torch.cuda.empty_cache()
            return [_generate(model, tok, [m], max_new_tokens)[0] for m in msgs_list]

LANG_RULE = {"en": "Reply only in English.",
             "bn": "Reply only in Bangla (Bengali script). Keep course codes, names, room numbers, times and emails exactly as written.",
             "banglish": "Reply only in Banglish (Bangla written with English letters, for example 'Exam Office e jogajog koro'). "
                         "Keep course codes, names, room numbers, times and emails exactly as written."}

# ---------------- S1: LLM-only (closed book) ----------------
LLM_ONLY_SYSTEM = ("You are the student-support assistant of Metropolitan University (MU), Sylhet, Bangladesh. "
                   "Answer the student's question in 1-3 short sentences. If you do not know the answer for certain, "
                   "say that you don't know. ")
def prep_llm_only(q):
    lang = rag.detect_language(q)
    st = dict(q=q, lang=lang, fact=None, conf=None, status="llm_only", route="")
    return st, [{"role": "system", "content": LLM_ONLY_SYSTEM + LANG_RULE[lang]}, {"role": "user", "content": q}]

def finish_llm_only(st, gen):
    gen = (gen or "").strip()
    return dict(answer=gen, declined=looks_like_refusal(gen), fact=None, conf=None, guard="", mode="llm", lang=st["lang"])

# ---------------- S3: final RAG (verified answer, LLM rewrite behind the guard) ----------------
RAG_SYSTEM = ("You are the official student-support assistant of Metropolitan University (MU), Sylhet. "
              "Rewrite the verified answer as a short, natural reply to the student's question. "
              "Use ONLY the verified answer. Keep EVERY number, date, time, amount, course code, name, room, phone number "
              "and email exactly as written, and do not add anything new. ")
def prep_rag(q):
    r = rag.retrieve(q, FINAL_SYSTEM)
    rule = RULES[FINAL]
    lang = r["lang"]
    st = dict(q=q, lang=lang, fact=r["ranking"][0][0] if r["ranking"] else None, conf=r["conf"], status=r["status"])
    if not MURAG.decide(r, rule):
        st["route"] = refusal_route(q, r["status"])
        return st, None
    st["route"] = ""
    verified = rag.verified_answer(st["fact"], lang)
    st["verified"] = verified
    if (not USE_LLM_REWRITE) or KIND[st["fact"]] != "answer" or len(verified) > REWRITE_MAX_CHARS:
        return st, None
    return st, [{"role": "system", "content": RAG_SYSTEM + LANG_RULE[lang]},
                {"role": "user", "content": f"Verified answer:\n{verified}\n\nStudent question: {q}"}]

def finish_rag(st, gen):
    if st["route"]:
        return dict(answer=REFUSE_MSG[st["route"]][st["lang"]], declined=True, fact=st["fact"], conf=st["conf"],
                    guard="", mode=f"refused:{st['route']}", lang=st["lang"])
    declined = KIND[st["fact"]] == "unknown_refer"
    if gen is None:
        return dict(answer=st["verified"], declined=declined, fact=st["fact"], conf=st["conf"], guard="", mode="verified", lang=st["lang"])
    probs = rag.guard(gen, st["verified"], st["lang"])
    if probs:
        return dict(answer=st["verified"], declined=declined, fact=st["fact"], conf=st["conf"], guard=";".join(probs),
                    mode="verified (guard fallback)", lang=st["lang"])
    return dict(answer=gen, declined=declined, fact=st["fact"], conf=st["conf"], guard="", mode="llm (grounded)", lang=st["lang"])

PIPELINES = {"S1. LLM-only": (prep_llm_only, finish_llm_only),
             "S3. RAG final": (prep_rag, finish_rag)}

def answer(q, system="S3. RAG final"):
    prep, finish = PIPELINES[system]
    st, msgs = prep(q)
    return finish(st, chat_generate_batch(llm, tok, [msgs])[0] if msgs else None)

for q in ["CSE 58 A er monday class ki?", "সাপ্লি পরীক্ষার রেজিস্ট্রেশন কোথায় করব?", "What is the routine of CSE 70 A?",
          "Who won the 2022 FIFA World Cup?", "retake fee koto?"]:
    a = answer(q)
    print(f"Q: {q}\n  RAG: {a['answer'][:220]}\n  [{a['mode']}; conf={a['conf']:.3f}]\n")

## 9. End-to-end evaluation on the test split (same questions for every system)
An answer is **correct** only when it states every number and email of the gold answer and at least half of its rare key words, keeps the yes/no polarity, adds no number that is not in the gold answer, and is written in the user's script. If the gold answer is itself "I don't know, contact the office", a refusal counts as correct. For out-of-domain questions only a refusal is correct. Checkpoints are saved, so re-running this cell continues where it stopped.

In [ ]:
def judge(row, out):
    gold = row.gold
    if row.expected == "refuse":
        return "correct_refusal" if out["declined"] else "answered_ood"
    kinds = {KIND[g] for g in gold}
    if "unknown_refer" in kinds:
        return "correct" if out["declined"] or out.get("fact") in gold else "wrong"
    if out["declined"]:
        return "abstain"
    ans = out["answer"]
    lang = row.language
    gold_nums = set().union(*[numbers(FACT.at[g, c]) for g in gold for c in ["answer_en", "answer_bn", "answer_banglish"]])
    if numbers(ans) - gold_nums:
        return "wrong"                                 # hallucinated / wrong number
    if not script_ok(ans, lang, FACT.at[gold[0], mu_rag.LANG_COL[lang]]):
        return "wrong_language"
    ok = any(rag.covers(ans, FACT.at[g, c], l) for g in gold
             for c, l in [(mu_rag.LANG_COL[lang], lang), ("answer_en", "en")])
    return "correct" if ok else "wrong"

def evaluate(frame, name, chunk=4 * GEN_BATCH):
    prep, finish = PIPELINES[name]
    import hashlib    # the checkpoint is only reused when the settings are identical
    sig = hashlib.md5(json.dumps([name, RULES.get(FINAL), FINAL, LLM_MODEL, USE_LLM_REWRITE, REWRITE_MAX_CHARS, MOCK,
                                  sorted(frame.eval_id)], default=str).encode()).hexdigest()[:8]
    ckpt = f"outputs/partial_{name.split('.')[0]}_{sig}.csv"
    rows = pd.read_csv(ckpt, keep_default_na=False).to_dict("records") if os.path.exists(ckpt) else []
    rows = [r for r in rows if r["eval_id"] in set(frame.eval_id)]
    done = {r["eval_id"] for r in rows}
    pending = [r for r in frame.itertuples() if r.eval_id not in done]
    if done: print(f"[{name}] resuming: {len(done)} already done")
    start = time.time()
    for c in range(0, len(pending), chunk):
        part = pending[c:c + chunk]; t0 = time.time()
        prepared = [prep(r.question) for r in part]
        need = sorted([j for j, (_, m) in enumerate(prepared) if m is not None], key=lambda j: len(prepared[j][1][-1]["content"]))
        gens = {}
        for b in range(0, len(need), GEN_BATCH):
            idx = need[b:b + GEN_BATCH]
            for j, g in zip(idx, chat_generate_batch(llm, tok, [prepared[j][1] for j in idx])):
                gens[j] = g
        dt = (time.time() - t0) / len(part)
        for j, r in enumerate(part):
            out = finish(prepared[j][0], gens.get(j))
            res = judge(r, out)
            rows.append(dict(system=name, eval_id=r.eval_id, language=r.language, type=r.type, category=r.category,
                             expected=r.expected, question=r.question, gold="|".join(r.gold), answer=out["answer"],
                             fact=out.get("fact") or "", fact_hit=(out.get("fact") in r.gold) if r.gold else "",
                             conf=out["conf"] if out["conf"] is not None else "", mode=out["mode"], guard=out["guard"],
                             declined=out["declined"], outcome=res, seconds=round(dt, 3)))
        pd.DataFrame(rows).to_csv(ckpt, index=False)
        k = c + len(part); per = (time.time() - start) / k
        print(f"[{name}] {len(rows)}/{len(frame)} | {per:.1f}s per question | ~{per * (len(pending) - k) / 60:.0f} min left", flush=True)
    return pd.DataFrame(rows)

t0 = time.time()
res = pd.concat([evaluate(test, name) for name in PIPELINES], ignore_index=True)
# S2 (old dense RAG) and S3 without LLM, from the retrieval runs (no generation needed)
for rule_name, label in [("B. Dense bge-m3 (old RAG)", "S2. RAG baseline (dense, verified answer)"),
                         (FINAL, "S3b. RAG final (verified answer, no LLM)")]:
    _, d = decision_summary(test_runs, rule_name, RULES[rule_name])
    d = d.assign(system=label, fact_hit=[t in g for t, g in zip(d.top, d.gold)])
    res = pd.concat([res, d[["system", "eval_id", "language", "type", "expected", "outcome", "fact_hit"]]], ignore_index=True)
res.to_csv("outputs/end_to_end_results.csv", index=False)
print(f"done in {(time.time() - t0) / 60:.1f} min")

def summary(d):
    ind, ood = d[d.expected == "answer"], d[d.expected == "refuse"]
    c = Counter(ind.outcome); o = Counter(ood.outcome)
    wrong = c["wrong"] + c["wrong_language"]
    return pd.Series({"In-domain n": len(ind), "Answer accuracy": c["correct"] / len(ind),
                      "Wrong answers": wrong / len(ind), "Abstained": c["abstain"] / len(ind),
                      "Precision when answering": c["correct"] / max(1, c["correct"] + wrong),
                      "OOD n": len(ood), "OOD correctly refused": o["correct_refusal"] / max(1, len(ood)),
                      "Overall (in-domain + OOD) accuracy": (c["correct"] + o["correct_refusal"]) / max(1, len(d))})

order = ["S1. LLM-only", "S2. RAG baseline (dense, verified answer)", "S3b. RAG final (verified answer, no LLM)", "S3. RAG final"]
main = pd.DataFrame({name: summary(g) for name, g in res.groupby("system")}).reindex(columns=order).round(3)
print("=== MAIN RESULT: LLM-only vs RAG (test split) ===")
display(main); main.to_csv("outputs/main_result.csv")
print("=== Answer accuracy by language (in-domain) ===")
display(res[res.expected == "answer"].assign(ok=lambda d: d.outcome == "correct")
        .pivot_table(index="language", columns="system", values="ok", aggfunc="mean").reindex(columns=order).round(3))
print("=== Answer accuracy by question type ===")
display(res.assign(ok=lambda d: d.outcome.isin(["correct", "correct_refusal"]))
        .pivot_table(index="type", columns="system", values="ok", aggfunc="mean").reindex(columns=order).round(3))
s3 = res[res.system == "S3. RAG final"]
print("RAG answer modes:", s3["mode"].value_counts().to_dict())
print("Guard triggers:", Counter(g.split(":")[0] for gs in s3["guard"] if gs for g in gs.split(";")))

## 10. Statistical significance (paired tests on the same test questions)
Every system answered the **same** test questions, so the comparisons are **paired**:
* **Exact McNemar test** for the paired binary outcome (correct / not correct per question). This is the appropriate paired test for 0/1 outcomes; a paired t-test needs continuous scores.
* **Paired bootstrap 95% CI** of the accuracy difference (the same questions are resampled for both systems, 5,000 times).
* **Holm correction** for running several comparisons.
* Each comparison is reported **overall** (in-domain + out-of-domain) and for **in-domain answer accuracy** only.

In [ ]:
from scipy.stats import binomtest

def ok_frame(name):
    d = res[res.system == name].drop_duplicates("eval_id").set_index("eval_id")
    return d["outcome"].isin(["correct", "correct_refusal"]), d["expected"]

def mcnemar_exact(a_ok, b_ok):
    b = int((b_ok & ~a_ok).sum()); c = int((a_ok & ~b_ok).sum())       # b: only B correct, c: only A correct
    return b, c, (binomtest(b, b + c, 0.5).pvalue if b + c else 1.0)

def paired_bootstrap(a_ok, b_ok, n_boot=5000, seed=SEED):
    a, b = a_ok.values.astype(float), b_ok.values.astype(float)
    rng_b = np.random.default_rng(seed)
    idx = rng_b.integers(0, len(a), (n_boot, len(a)))
    diffs = b[idx].mean(axis=1) - a[idx].mean(axis=1)
    return np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

def holm(pvals):
    m = len(pvals); order = np.argsort(pvals); adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * pvals[i])); adj[i] = running
    return adj

PAIRS = [("S1. LLM-only", "S3. RAG final"),                                   # main thesis comparison
         ("S1. LLM-only", "S2. RAG baseline (dense, verified answer)"),
         ("S2. RAG baseline (dense, verified answer)", "S3b. RAG final (verified answer, no LLM)"),
         ("S3b. RAG final (verified answer, no LLM)", "S3. RAG final")]       # effect of the LLM rewrite step
rows = []
for scope in ["overall (in-domain + OOD)", "in-domain only"]:
    for A, B in PAIRS:
        a_ok, a_exp = ok_frame(A); b_ok, _ = ok_frame(B)
        ids = a_ok.index.intersection(b_ok.index)
        if scope == "in-domain only":
            ids = ids[(a_exp.loc[ids] == "answer").values]
        a_ok, b_ok = a_ok.loc[ids], b_ok.loc[ids]
        nb_, nc_, p = mcnemar_exact(a_ok, b_ok)
        lo, hi = paired_bootstrap(a_ok, b_ok)
        rows.append({"scope": scope, "A": A, "B": B, "n questions": len(ids),
                     "accuracy A": a_ok.mean(), "accuracy B": b_ok.mean(), "difference (B - A)": b_ok.mean() - a_ok.mean(),
                     "95% CI low": lo, "95% CI high": hi, "only B correct": nb_, "only A correct": nc_, "McNemar p": p})
sig = pd.DataFrame(rows)
sig["p (Holm-adjusted)"] = holm(sig["McNemar p"].values)
sig["significant (p<0.05)"] = sig["p (Holm-adjusted)"] < 0.05
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
show = sig.copy()
for c in ["accuracy A", "accuracy B", "difference (B - A)", "95% CI low", "95% CI high"]:
    show[c] = show[c].round(3)
for c in ["McNemar p", "p (Holm-adjusted)"]:
    show[c] = show[c].map(lambda x: f"{x:.2e}")          # very small p-values stay visible (not rounded to 0)
display(show)
sig.to_csv("outputs/significance_tests.csv", index=False)
pd.reset_option("display.float_format")

print("\n95% bootstrap CI of each system's overall accuracy:")
boot = np.random.default_rng(SEED)
for name in order:
    s_ok = ok_frame(name)[0].values.astype(float)
    accs = s_ok[boot.integers(0, len(s_ok), (5000, len(s_ok)))].mean(axis=1)
    print(f"  {name:45s} {s_ok.mean():.3f}  (95% CI {np.percentile(accs, 2.5):.3f}-{np.percentile(accs, 97.5):.3f})")

m = sig.iloc[0]
print(f"\nThesis sentence: On {m['n questions']} paired test questions, RAG answered {m['accuracy B']:.1%} correctly versus "
      f"{m['accuracy A']:.1%} for the LLM alone (difference {m['difference (B - A)']:+.1%}, 95% CI "
      f"{m['95% CI low']:+.1%} to {m['95% CI high']:+.1%}; exact McNemar p = {m['McNemar p']:.2g}, Holm-adjusted p = {m['p (Holm-adjusted)']:.2g}).")

## 11. Error analysis and a sample for human evaluation

In [ ]:
err = s3[~s3.outcome.isin(["correct", "correct_refusal"])]
print("RAG final - remaining errors:", err["outcome"].value_counts().to_dict())
display(err[["language", "type", "question", "gold", "fact", "outcome", "answer"]].head(25))
err.to_csv("outputs/rag_errors.csv", index=False)
llm_wrong = res[(res.system == "S1. LLM-only") & (res.outcome.isin(["wrong", "answered_ood"]))]
print(f"\nLLM-only wrong/hallucinated answers: {len(llm_wrong)} - examples:")
display(llm_wrong[["language", "question", "answer"]].head(10))
# 100 random answers of both systems for 2 human raters (fill the empty columns, then compute Cohen's kappa)
hs = res[res.system.isin(["S1. LLM-only", "S3. RAG final"])].sample(min(200, len(res)), random_state=SEED)
hs.assign(rater1_correct="", rater2_correct="")[["system", "language", "question", "answer", "rater1_correct", "rater2_correct"]] \
  .to_csv("outputs/human_eval_sample.csv", index=False)
print("Saved outputs/human_eval_sample.csv for the human evaluation")

## 12. Figures for the thesis
All figures are saved in `outputs/figures/` as **PNG (300 dpi)** for Word and **PDF** for LaTeX, together with `captions.md` (suggested captions with the numbers of this run).
* **Main text (`main_fig1` … `main_fig6`):** the pipeline, overall accuracy with 95% CI, in-domain outcomes (correct / wrong / "I don't know"), the retrieval ablation, accuracy by language, and coverage vs precision.
* **Appendix (`appendix_figA1` … `A3`):** out-of-domain refusal, accuracy by question type, and typo robustness.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

FIG_DIR = "outputs/figures"; os.makedirs(FIG_DIR, exist_ok=True)
# validated categorical palette (fixed order; checked for colour-blind separation) + neutral ink colours
SYS_COLOR = {"S1. LLM-only": "#2a78d6", "S2. RAG baseline (dense, verified answer)": "#eb6834",
             "S3b. RAG final (verified answer, no LLM)": "#1baf7a", "S3. RAG final": "#eda100"}
SHORT = {"S1. LLM-only": "LLM-only", "S2. RAG baseline (dense, verified answer)": "RAG baseline\n(dense only)",
         "S3b. RAG final (verified answer, no LLM)": "RAG final\n(no LLM rewrite)", "S3. RAG final": "RAG final\n(with LLM)"}
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
GOOD, CRIT, NEUTRAL = "#0ca30c", "#d03b3b", "#c3c2b7"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
                     "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.edgecolor": AXIS,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID,
                     "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False})

def save(fig, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

def bar_labels(ax, bars, fmt="{:.0%}", inside=False):
    for b in bars:
        v = b.get_height()
        if np.isnan(v): continue
        ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, v), xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8.5, color=INK)

def ok_vec(name):
    d = res[res.system == name].drop_duplicates("eval_id")
    return d["outcome"].isin(["correct", "correct_refusal"]).values.astype(float)

# ---------- Figure 1: overall accuracy with 95% bootstrap CI ----------
rng_f = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(7, 4))
means, los, his = [], [], []
for name in order:
    s_ok = ok_vec(name)
    boots = s_ok[rng_f.integers(0, len(s_ok), (5000, len(s_ok)))].mean(axis=1)
    means.append(s_ok.mean()); los.append(np.percentile(boots, 2.5)); his.append(np.percentile(boots, 97.5))
x = np.arange(len(order))
bars = ax.bar(x, means, width=0.6, color=[SYS_COLOR[n] for n in order], edgecolor="white", linewidth=2)
ax.errorbar(x, means, yerr=[np.array(means) - los, np.array(his) - means], fmt="none", ecolor=INK2, elinewidth=1.2, capsize=4)
for xi, m, h in zip(x, means, his):
    ax.annotate(f"{m:.1%}", (xi, h), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9, color=INK)
ax.set_xticks(x, [SHORT[n] for n in order]); ax.set_ylim(0, 1.08); ax.set_ylabel("Overall accuracy")
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_title(f"Overall accuracy on {int(len(ok_vec(order[0])))} test questions (95% bootstrap CI)")
save(fig, "main_fig2_overall_accuracy")

# ---------- Figure 2: what happens to in-domain questions (correct / wrong / "I don't know") ----------
fig, ax = plt.subplots(figsize=(7.5, 3.6))
parts = {"Correct": [], "Wrong": [], "Said \"I don't know\"": []}
for name in order:
    d = res[(res.system == name) & (res.expected == "answer")]
    c = Counter(d.outcome); n = len(d)
    parts["Correct"].append(c["correct"] / n); parts["Wrong"].append((c["wrong"] + c["wrong_language"]) / n)
    parts["Said \"I don't know\""].append(c["abstain"] / n)
y = np.arange(len(order))[::-1]; left = np.zeros(len(order))
for label, col in zip(parts, [GOOD, CRIT, NEUTRAL]):
    vals = np.array(parts[label])
    ax.barh(y, vals, left=left, color=col, edgecolor="white", linewidth=2, height=0.6, label=label)
    for yi, l, v in zip(y, left, vals):
        if v >= 0.06:
            ax.text(l + v / 2, yi, f"{v:.0%}", ha="center", va="center", fontsize=8.5, color="white" if col != NEUTRAL else INK)
    left += vals
ax.set_yticks(y, [SHORT[n].replace("\n", " ") for n in order]); ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
ax.set_title("In-domain questions: correct, wrong and \"I don't know\" answers")
save(fig, "main_fig3_indomain_outcomes")

# ---------- Figure 3: out-of-domain questions correctly refused ----------
fig, ax = plt.subplots(figsize=(7, 3.6))
vals = [(res[(res.system == n) & (res.expected == "refuse")].outcome == "correct_refusal").mean() for n in order]
bars = ax.bar(x, vals, width=0.6, color=[SYS_COLOR[n] for n in order], edgecolor="white", linewidth=2)
bar_labels(ax, bars); ax.set_xticks(x, [SHORT[n] for n in order]); ax.set_ylim(0, 1.1)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.set_ylabel("Correctly refused")
ax.set_title("Out-of-domain and unknown-entity questions correctly refused")
save(fig, "appendix_figA1_ood_refusal")

# ---------- Figure 4: retrieval ablation ----------
ab = ablation.copy()
fig, ax = plt.subplots(figsize=(8, 3.8))
xi = np.arange(len(ab)); w = 0.38
b1 = ax.bar(xi - w / 2, ab["R@1"], w, color="#2a78d6", edgecolor="white", linewidth=2, label="All in-domain questions")
b2 = ax.bar(xi + w / 2, ab["R@1 hard near-duplicates"], w, color="#eb6834", edgecolor="white", linewidth=2,
            label="Hard near-duplicates (58 A vs 58 B, 1307 vs 1308)")
bar_labels(ax, b1, "{:.2f}"); bar_labels(ax, b2, "{:.2f}")
ax.set_xticks(xi, [i.split(". ", 1)[1].replace(" (", "\n(").replace(" + ", "\n+ ") for i in ab.index], fontsize=8.5)
ax.set_ylim(0, 1.12); ax.set_ylabel("Recall@1 (correct fact ranked first)")
ax.legend(loc="upper left", fontsize=8.5); ax.set_title("Retrieval ablation: what each component adds (test split)")
save(fig, "main_fig4_retrieval_ablation")

# ---------- Figure 5: accuracy by language ----------
langs = ["en", "bn", "banglish"]; LANG_NAME = {"en": "English", "bn": "Bangla", "banglish": "Banglish"}
fig, ax = plt.subplots(figsize=(8, 3.8)); w = 0.2
for k, name in enumerate(order):
    d = res[(res.system == name) & (res.expected == "answer")]
    vals = [(d[d.language == l].outcome == "correct").mean() for l in langs]
    bars = ax.bar(np.arange(3) + (k - 1.5) * w, vals, w, color=SYS_COLOR[name], edgecolor="white", linewidth=1.5,
                  label=SHORT[name].replace("\n", " "))
    bar_labels(ax, bars)
ax.set_xticks(np.arange(3), [LANG_NAME[l] for l in langs]); ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.set_ylabel("In-domain answer accuracy")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=4, fontsize=8.5)
ax.set_title("Answer accuracy by question language")
save(fig, "main_fig5_accuracy_by_language")

# ---------- Figure 6: accuracy by question type (heatmap) ----------
tab = (res.assign(ok=lambda d: d.outcome.isin(["correct", "correct_refusal"]))
          .pivot_table(index="type", columns="system", values="ok", aggfunc="mean").reindex(columns=order))
fig, ax = plt.subplots(figsize=(8, 0.45 * len(tab) + 1.6))
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("blue", ["#f4f8fd", "#86b6ef", "#2a78d6", "#104281"])
im = ax.imshow(tab.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
for i in range(tab.shape[0]):
    for j in range(tab.shape[1]):
        v = tab.values[i, j]
        ax.text(j, i, "" if np.isnan(v) else f"{v:.0%}", ha="center", va="center", fontsize=8.5,
                color="white" if v > 0.6 else INK)
ax.set_xticks(range(len(order)), [SHORT[n] for n in order], fontsize=8.5)
ax.set_yticks(range(len(tab)), [t.replace("_", " ") for t in tab.index], fontsize=8.5)
ax.grid(False); ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, format=matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Accuracy by question type (correct answer, or correct refusal for out-of-domain types)")
save(fig, "appendix_figA2_accuracy_by_type")

# ---------- Figure 7: typo robustness ----------
fig, ax = plt.subplots(figsize=(8, 3.6)); xi = np.arange(len(typo)); w = 0.38
b1 = ax.bar(xi - w / 2, typo["R@1 clean"], w, color="#2a78d6", edgecolor="white", linewidth=2, label="Clean questions")
b2 = ax.bar(xi + w / 2, typo["R@1 typos"], w, color="#eb6834", edgecolor="white", linewidth=2, label="Same questions with typos")
bar_labels(ax, b1, "{:.2f}"); bar_labels(ax, b2, "{:.2f}")
ax.set_xticks(xi, [i.split(". ", 1)[1].replace(" (", "\n(").replace(" + ", "\n+ ") for i in typo.index], fontsize=8.5)
ax.set_ylim(0, 1.12); ax.set_ylabel("Recall@1"); ax.legend(loc="upper left", fontsize=8.5)
ax.set_title("Robustness to spelling mistakes (Recall@1)")
save(fig, "appendix_figA3_typo_robustness")

# ---------- Figure 8: coverage vs precision (how the refusal threshold trades answers for safety) ----------
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for name, col in [("B. Dense bge-m3 (old RAG)", "#eb6834"), (FINAL, "#1baf7a")]:
    d = test_runs[(test_runs.system == name) & (test_runs.expected == "answer")].reset_index(drop=True)
    okm = ((d.status == "ok") & d.top.map(lambda t: isinstance(t, str))).values
    hit = np.array([isinstance(t, str) and t in g for t, g in zip(d.top, d.gold)])
    cov, prec = [], []
    for t in np.unique(np.quantile(d.conf, np.linspace(0, 0.98, 60))):
        acc = okm & (d.conf.values >= t)
        if acc.sum() == 0: continue
        cov.append(acc.mean()); prec.append(hit[acc].mean())
    lab = "RAG final (reranker score)" if name == FINAL else "RAG baseline (dense cosine)"
    ax.plot(cov, prec, color=col, linewidth=2, label=lab)
    acc = accepted_mask(d, RULES[name])
    ax.plot(acc.mean(), hit[acc].mean() if acc.any() else np.nan, "o", markersize=8, color=col,
            markeredgecolor="white", markeredgewidth=1.5)
    ax.annotate("tuned rule (all paths)", (acc.mean(), hit[acc].mean() if acc.any() else 0), xytext=(-60, -18),
                textcoords="offset points", fontsize=8, color=INK2, arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))
ax.set_xlabel("Coverage (share of in-domain questions answered)"); ax.set_ylabel("Precision (answers that are correct)")
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlim(0, 1.02); ax.set_ylim(0.5, 1.02); ax.grid(axis="both"); ax.legend(loc="lower left", fontsize=8.5)
ax.set_title("Coverage vs precision as the threshold changes (test split)")
save(fig, "main_fig6_coverage_precision")

# ---------- Figure 9: pipeline diagram ----------
fig, ax = plt.subplots(figsize=(11, 3.2)); ax.axis("off"); ax.set_xlim(0, 11); ax.set_ylim(0, 3.2)
steps = ["Question\n(EN / BN /\nBanglish)", "Language\ndetection +\nspell fix", "Entity check\n(batch, code,\nbus, other uni)",
         "Hybrid search\nbge-m3 + BM25\n+ char (RRF)", "Reranker\nbge-reranker\n-v2-m3", "Answer or\nrefuse?\n(tuned rule)",
         "Verified answer\n(+ LLM rewrite\nbehind guard)"]
for i, s_ in enumerate(steps):
    x0 = 0.15 + i * 1.55
    ax.add_patch(FancyBboxPatch((x0, 1.05), 1.3, 1.3, boxstyle="round,pad=0.02,rounding_size=0.12",
                                facecolor="#cde2fb" if i not in (5,) else "#fde7c2", edgecolor="#2a78d6" if i != 5 else "#eda100", lw=1.2))
    ax.text(x0 + 0.65, 1.7, s_, ha="center", va="center", fontsize=8.5, color=INK)
    if i < len(steps) - 1:
        ax.annotate("", xy=(x0 + 1.55, 1.7), xytext=(x0 + 1.3, 1.7), arrowprops=dict(arrowstyle="->", color=INK2, lw=1.2))
ax.text(0.15 + 5 * 1.55 + 0.65, 0.55, "low score / unknown entity →\n\"I don't know\" + Exam Office / MU office",
        ha="center", va="center", fontsize=8, color=INK2)
ax.annotate("", xy=(0.15 + 5 * 1.55 + 0.65, 0.8), xytext=(0.15 + 5 * 1.55 + 0.65, 1.05), arrowprops=dict(arrowstyle="->", color=INK2, lw=1.0))
ax.set_title("Final RAG pipeline of the MU student-support chatbot", loc="left")
save(fig, "main_fig1_pipeline")

# ---------- suggested captions with the real numbers of this run ----------
m3 = {n: summary(res[res.system == n]) for n in order}
L, R = "S1. LLM-only", "S3. RAG final"
ab_r1 = ablation["R@1"]; hard = ablation["R@1 hard near-duplicates"]
cap = f"""# Suggested figure captions (numbers from this run)

## Main text
- **Figure 1 (main_fig1_pipeline):** Pipeline of the proposed RAG chatbot: language detection, entity check, hybrid retrieval (bge-m3 + BM25 + character n-grams, RRF), cross-encoder reranking, a tuned answer/refuse rule and a guarded answer.
- **Figure 2 (main_fig2_overall_accuracy):** Overall accuracy of the four systems on {int(m3[L]['In-domain n'] + m3[L]['OOD n'])} test questions with 95% bootstrap confidence intervals. RAG final: {m3[R]['Overall (in-domain + OOD) accuracy']:.1%}; LLM-only: {m3[L]['Overall (in-domain + OOD) accuracy']:.1%}.
- **Figure 3 (main_fig3_indomain_outcomes):** Outcome of in-domain questions. LLM-only gave wrong answers to {m3[L]['Wrong answers']:.1%} of them; RAG final to {m3[R]['Wrong answers']:.1%}, and said "I don't know" to {m3[R]['Abstained']:.1%}.
- **Figure 4 (main_fig4_retrieval_ablation):** Retrieval ablation (Recall@1). Dense only: {ab_r1.iloc[1]:.2f} overall / {hard.iloc[1]:.2f} on hard near-duplicates; final retriever: {ab_r1.iloc[-1]:.2f} / {hard.iloc[-1]:.2f}.
- **Figure 5 (main_fig5_accuracy_by_language):** In-domain answer accuracy by question language (English, Bangla, Banglish).
- **Figure 6 (main_fig6_coverage_precision):** Coverage versus precision when the confidence threshold is varied; the dot marks the rule tuned on the dev split.

## Appendix
- **Figure A1 (appendix_figA1_ood_refusal):** Out-of-domain and unknown-entity questions correctly refused (RAG final: {m3[R]['OOD correctly refused']:.1%}; LLM-only: {m3[L]['OOD correctly refused']:.1%}).
- **Figure A2 (appendix_figA2_accuracy_by_type):** Accuracy by question type.
- **Figure A3 (appendix_figA3_typo_robustness):** Recall@1 on clean questions and on the same questions with injected spelling mistakes.
"""
open(f"{FIG_DIR}/captions.md", "w", encoding="utf-8").write(cap)
print(cap)
print("Saved figures (PNG 300 dpi + PDF) and captions.md in", FIG_DIR, ":", sorted(os.listdir(FIG_DIR)))

## 13. LangChain integration (same pipeline as a LangChain retriever and chain)
LangChain is used here as the **standard interface**: the retriever (`BaseRetriever`) and the chatbot (`Runnable`) can be plugged into any LangChain application. It does **not** change the accuracy, because it calls exactly the same pipeline that was evaluated above; the cell checks this on 20 questions.

In [ ]:
# LangChain wrapper around the SAME pipeline (retriever + decision + guarded answer).
# It does not change any result above; it makes the chatbot usable as a standard LangChain component.
try:
    from typing import List
    from langchain_core.retrievers import BaseRetriever
    from langchain_core.documents import Document
    from langchain_core.runnables import RunnableLambda

    class MURetriever(BaseRetriever):
        """LangChain retriever: hybrid search + entity filter + reranker of mu_rag (one Document per fact)."""
        k: int = 3

        def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
            r = rag.retrieve(query, FINAL_SYSTEM, k=self.k)
            accepted = MURAG.decide(r, RULES[FINAL])
            return [Document(page_content=rag.verified_answer(f, r["lang"]),
                             metadata={"fact_id": f, "score": round(s, 4), "language": r["lang"], "rank": i + 1,
                                       "answerable": accepted, "source": str(FACT.at[f, "source"])})
                    for i, (f, s) in enumerate(r["ranking"])]

    mu_retriever = MURetriever(k=3)
    # full chatbot as a LangChain Runnable: question -> retrieval -> answer/refuse -> guarded LLM answer
    mu_chain = RunnableLambda(lambda q: answer(q)["answer"]).with_config(run_name="MU-RAG-chatbot")

    q = "supply exam er registration kothay korbo?"
    print("Retriever:", [(d.metadata["fact_id"], d.metadata["score"], d.metadata["answerable"]) for d in mu_retriever.invoke(q)])
    print("Chain answer:", mu_chain.invoke(q)[:200])
    # consistency check: the LangChain chain gives exactly the same answers as the pipeline evaluated above
    sample_q = list(test["question"].sample(min(20, len(test)), random_state=SEED))
    same = sum(mu_chain.invoke(s) == answer(s)["answer"] for s in sample_q)
    print(f"LangChain chain == evaluated pipeline on {same}/{len(sample_q)} sample questions")
except Exception as e:
    print("LangChain section skipped (does not affect any result above):", type(e).__name__, str(e)[:200])

## 14. Try it and save the settings for the web app

In [ ]:
for q in ["CSE te koto credit lage?", "সাপ্লি পরীক্ষার ফি কত?", "rikabibazar theke bus koytay?", "What is the WiFi password?",
          "CSE 0613 1308 er credit koto?", "Who is the head of the EEE department?", "What is the admission fee at BRAC University?",
          "rima mam ki class nen?", "srabonti maam ki class nen?", "রিমা ম্যাম কী পড়ান?", "SRR ke?",
          "supply er jonno registration kuthay korte hoy?", "সাপ্লির রেজিস্ট্রেশন কোথায় করতে হয়?",
          "ek time a koto ta supply dewa jay?", "how many supplies can be tested at a time??"]:
    a = answer(q)
    print(f"Q: {q}\nA: {a['answer'][:300]}\n   ({a['mode']}, confidence={a['conf'] if a['conf'] is not None else 0:.3f})\n")

json.dump({"rule": RULES[FINAL], "final_system": FINAL_SYSTEM, "embed_model": EMBED_MODEL, "rerank_model": RERANK_MODEL,
           "llm": LLM_MODEL, "n_rerank": N_RERANK, "use_llm_rewrite": USE_LLM_REWRITE, "rewrite_max_chars": REWRITE_MAX_CHARS,
           "kb_csv": os.path.abspath(find("MUBot_v5_KnowledgeBase.csv")), "facts_csv": os.path.abspath(find("MUBot_v5_Facts.csv"))},
          open("outputs/config.json", "w"), indent=2)
print("Saved:", sorted(os.listdir("outputs")))

## 15. Live chatbot web app (Streamlit), optional
1. In Kaggle, open *Add-ons → Secrets*, add `NGROK_TOKEN` (a free token from ngrok.com) and attach it to this notebook.
2. Run the cells below. The last one prints a **PUBLIC APP URL** that opens on any phone or computer.

The app uses exactly the same `mu_rag.py` pipeline and the rule tuned above.

In [ ]:
%%writefile app.py
import os, re, csv, json, glob, time, html
from datetime import datetime

import numpy as np
import pandas as pd
import streamlit as st

from mu_rag import MURAG, REFUSE_MSG, refusal_route

st.set_page_config(page_title="MU Student Support Chatbot", page_icon="🎓", layout="wide",
                   initial_sidebar_state="expanded")
OUT_DIR = os.environ.get("MU_OUT_DIR", "outputs")
CFG = json.load(open(os.path.join(OUT_DIR, "config.json"), encoding="utf-8"))
RULE = CFG["rule"]
FEEDBACK = os.path.join(OUT_DIR, "feedback_log.csv")
LANG_LABEL = {"en": "English", "bn": "বাংলা", "banglish": "Banglish"}
LANG_FLAG = {"en": "🇬🇧", "bn": "🇧🇩", "banglish": "🔤"}
# (icon, short title, question) for the welcome cards and the sidebar
SAMPLES = [("📅", "Class routine", "CSE 58 A er monday class ki?"),
           ("📝", "Supplementary exam", "সাপ্লি পরীক্ষার রেজিস্ট্রেশন কোথায় করব?"),
           ("💳", "Fees", "retake fee koto?"),
           ("🎓", "Credits", "How many credits are needed for BSc in CSE?"),
           ("🚌", "Bus schedule", "rikabibazar theke bus koytay?"),
           ("🛡️", "Out of scope", "Who won the 2022 FIFA World Cup?")]
LANG_RULE = {"en": "Reply only in English.",
             "bn": "Reply only in Bangla (Bengali script). Keep course codes, names, room numbers, times and emails exactly as written.",
             "banglish": "Reply only in Banglish (Bangla written with English letters). Keep course codes, names, room numbers, times and emails exactly as written."}
RAG_SYSTEM = ("You are the official student-support assistant of Metropolitan University (MU), Sylhet. "
              "Rewrite the verified answer as a short, natural reply to the student's question. "
              "Use ONLY the verified answer. Keep EVERY number, date, time, amount, course code, name, room, phone number "
              "and email exactly as written, and do not add anything new. ")


# =============================== pipeline (unchanged) ===============================
@st.cache_resource(show_spinner="Loading models (first time only, 2-4 minutes)...")
def load_pipeline():
    import torch
    from sentence_transformers import SentenceTransformer
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
                              BitsAndBytesConfig)
    n_gpu = torch.cuda.device_count()
    dev_llm = "cuda:0" if n_gpu else "cpu"
    dev_ret = "cuda:1" if n_gpu > 1 else dev_llm
    kb = pd.read_csv(CFG["kb_csv"], keep_default_na=False)
    facts = pd.read_csv(CFG["facts_csv"], keep_default_na=False)
    emb = SentenceTransformer(CFG["embed_model"], device=dev_ret)
    if n_gpu: emb.half()
    def embed(texts):
        return emb.encode(list(texts), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    rtok = AutoTokenizer.from_pretrained(CFG["rerank_model"])
    rmod = AutoModelForSequenceClassification.from_pretrained(
        CFG["rerank_model"], torch_dtype=torch.float16 if n_gpu else torch.float32).to(dev_ret).eval()
    @torch.no_grad()
    def rerank(q, passages):
        enc = rtok([[q, p] for p in passages], padding=True, truncation=True, max_length=512, return_tensors="pt").to(dev_ret)
        return torch.sigmoid(rmod(**enc).logits.view(-1).float()).cpu().numpy()
    rag = MURAG(kb, facts, embed=embed, rerank=rerank, n_rerank=CFG.get("n_rerank", 20))
    llm = tok = None
    if CFG.get("use_llm_rewrite") and n_gpu:
        try:
            tok = AutoTokenizer.from_pretrained(CFG["llm"])
            if "7B" in CFG["llm"]:
                llm = AutoModelForCausalLM.from_pretrained(CFG["llm"], device_map={"": dev_llm}, quantization_config=BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)).eval()
            else:
                llm = AutoModelForCausalLM.from_pretrained(CFG["llm"], torch_dtype=torch.float16, device_map={"": dev_llm}).eval()
        except Exception:          # the app still works without the LLM: it shows the verified answers
            llm = tok = None
    return rag, llm, tok


def generate(llm, tok, msgs):
    import torch
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=200, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def respond(q):
    rag, llm, tok = load_pipeline()
    t0 = time.time()
    r = rag.retrieve(q, CFG.get("final_system", "final"))
    lang = r["lang"]
    info = dict(lang=lang, conf=r["conf"], margin=r["margin"], fact="", source="", mode="", status=r["status"])
    if not MURAG.decide(r, RULE):
        route = refusal_route(q, r["status"])
        info["mode"] = f"refused ({route})"
        return REFUSE_MSG[route][lang], info, time.time() - t0
    fid = r["ranking"][0][0]
    verified = rag.verified_answer(fid, lang)
    info.update(fact=fid, source=str(rag.facts.at[fid, "source"]))
    ans, info["mode"] = verified, "verified answer"
    if llm is not None and rag.kind[fid] == "answer" and len(verified) <= CFG.get("rewrite_max_chars", 350):
        gen = generate(llm, tok, [{"role": "system", "content": RAG_SYSTEM + LANG_RULE[lang]},
                                  {"role": "user", "content": f"Verified answer:\n{verified}\n\nStudent question: {q}"}])
        if not rag.guard(gen, verified, lang):
            ans, info["mode"] = gen, "LLM answer (checked against the verified fact)"
        else:
            info["mode"] = "verified answer (LLM rewrite rejected by the guard)"
    return ans, info, time.time() - t0


def log_feedback(q, a, info, rating):
    new = not os.path.exists(FEEDBACK)
    with open(FEEDBACK, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new: w.writerow(["time", "question", "answer", "language", "fact", "confidence", "mode", "rating"])
        w.writerow([datetime.now().isoformat(timespec="seconds"), q, a, info.get("lang"), info.get("fact"),
                    round(float(info.get("conf") or 0), 3), info.get("mode"), rating])


# =============================== look & feel ===============================
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Hind+Siliguri:wght@400;500;600&family=Poppins:wght@400;500;600;700&display=swap');
html, body, [class*="css"], .stMarkdown, .stChatMessage, button, input, textarea {
  font-family: 'Poppins', 'Hind Siliguri', sans-serif !important; }
.stApp { background: linear-gradient(180deg, #eef3fb 0%, #f7f9fc 260px, #f7f9fc 100%); }
.block-container { padding-top: 1.4rem; padding-bottom: 6rem; max-width: 1000px; }
#MainMenu, footer { visibility: hidden; }
header[data-testid="stHeader"] { background: transparent; }
[data-testid="stToolbar"] { display: none; }
[data-testid="stMain"] .stButton button p { white-space: normal !important; overflow: visible !important;
  text-overflow: clip !important; text-align: left; }

/* hero */
.hero { position: relative; overflow: hidden; border-radius: 22px; padding: 26px 30px 22px;
  background: linear-gradient(125deg, #0a2a66 0%, #123f93 45%, #1d6fd6 100%); color: #fff;
  box-shadow: 0 14px 34px rgba(13, 47, 110, .28); margin-bottom: 18px; }
.hero:after { content: ""; position: absolute; right: -60px; top: -60px; width: 240px; height: 240px;
  border-radius: 50%; background: radial-gradient(circle, rgba(255,255,255,.18), rgba(255,255,255,0) 70%); }
.hero .row { display: flex; align-items: center; gap: 16px; }
.hero .logo { width: 58px; height: 58px; border-radius: 16px; background: rgba(255,255,255,.16);
  display: flex; align-items: center; justify-content: center; font-size: 30px; border: 1px solid rgba(255,255,255,.3); }
.hero h1 { color: #fff; font-size: 1.65rem; font-weight: 700; margin: 0; line-height: 1.25; }
.hero p { margin: 4px 0 0; opacity: .9; font-size: .95rem; }
.chips { margin-top: 14px; display: flex; flex-wrap: wrap; gap: 8px; }
.chip { background: rgba(255,255,255,.14); border: 1px solid rgba(255,255,255,.3); padding: 4px 12px;
  border-radius: 999px; font-size: .78rem; }
.live { display: inline-flex; align-items: center; gap: 6px; }
.live:before { content: ""; width: 8px; height: 8px; border-radius: 50%; background: #3ddc84;
  box-shadow: 0 0 0 0 rgba(61,220,132,.7); animation: pulse 1.8s infinite; }
@keyframes pulse { 0% { box-shadow: 0 0 0 0 rgba(61,220,132,.7);} 70% { box-shadow: 0 0 0 8px rgba(61,220,132,0);} 100% { box-shadow: 0 0 0 0 rgba(61,220,132,0);} }

/* welcome */
.welcome { background: #fff; border-radius: 18px; padding: 18px 22px; border: 1px solid #e3e9f3;
  box-shadow: 0 4px 14px rgba(20, 50, 100, .06); margin-bottom: 12px; }
.welcome h3 { margin: 0 0 4px; font-size: 1.12rem; color: #0a2a66; }
.welcome p { margin: 0; color: #4b5563; font-size: .92rem; }

/* sample cards (buttons) */
div[data-testid="stHorizontalBlock"] .stButton button.sample, .sample-wrap .stButton button {
  text-align: left; }
[data-testid="stMain"] .stButton button { border-radius: 14px; border: 1px solid #dbe4f3; background: #fff; color: #0f2a5c;
  padding: 12px 14px; min-height: 64px; font-size: .88rem; transition: all .15s ease; white-space: normal;
  box-shadow: 0 2px 6px rgba(20,50,100,.05); }
[data-testid="stMain"] .stButton button:hover { border-color: #1d6fd6; color: #1d6fd6; transform: translateY(-2px);
  box-shadow: 0 8px 18px rgba(29,111,214,.15); }

/* chat bubbles */
div[data-testid="stChatMessage"] { background: #fff; border: 1px solid #e3e9f3; border-radius: 18px;
  padding: 14px 16px; box-shadow: 0 3px 12px rgba(20, 50, 100, .06); margin-bottom: 10px; }
div[data-testid="stChatMessage"]:has([aria-label="Chat message from user"]),
div[data-testid="stChatMessage"]:has(div[data-testid="stChatMessageAvatarUser"]) {
  background: linear-gradient(135deg, #e8f0fe, #dbe7fd); border-color: #c9daf8; }
.stChatMessage p { font-size: .97rem; line-height: 1.6; }

/* answer meta */
.meta { display: flex; flex-wrap: wrap; gap: 6px; align-items: center; margin-top: 8px; }
.badge { display: inline-flex; align-items: center; gap: 4px; padding: 2px 10px; border-radius: 999px;
  font-size: .74rem; font-weight: 500; border: 1px solid transparent; }
.b-lang { background: #eef3fb; color: #123f93; border-color: #d6e2f6; }
.b-ok { background: #e7f7ee; color: #12753e; border-color: #c4ead4; }
.b-llm { background: #f1ecff; color: #5b34c7; border-color: #ddd2fb; }
.b-ref { background: #fff4e5; color: #a95b00; border-color: #fbdcb0; }
.b-time { background: #f3f4f6; color: #4b5563; }
.conf { display: inline-flex; align-items: center; gap: 6px; font-size: .74rem; color: #4b5563; }
.conf .bar { width: 70px; height: 6px; border-radius: 6px; background: #e5e7eb; overflow: hidden; }
.conf .bar > span { display: block; height: 100%; border-radius: 6px; }
.src { margin-top: 8px; font-size: .78rem; color: #6b7280; background: #f8fafc; border-left: 3px solid #1d6fd6;
  padding: 6px 10px; border-radius: 6px; }

/* sidebar */
section[data-testid="stSidebar"] { background: linear-gradient(180deg, #0a2a66 0%, #0f3780 100%); }
section[data-testid="stSidebar"] * { color: #e8eefc !important; }
section[data-testid="stSidebar"] .stButton button { background: rgba(255,255,255,.08); border: 1px solid rgba(255,255,255,.18);
  border-radius: 12px; text-align: left; justify-content: flex-start; font-size: .83rem; padding: 8px 12px; }
section[data-testid="stSidebar"] .stButton button:hover { background: rgba(255,255,255,.18); border-color: rgba(255,255,255,.4); }
section[data-testid="stSidebar"] .stDownloadButton button { background: rgba(255,255,255,.08); border-radius: 12px;
  border: 1px solid rgba(255,255,255,.18); }
.side-brand { display: flex; align-items: center; gap: 10px; margin-bottom: 6px; }
.side-brand .lg { font-size: 26px; }
.side-brand b { font-size: 1.02rem; }
.stat { display: flex; gap: 8px; margin: 8px 0 4px; }
.stat div { flex: 1; background: rgba(255,255,255,.08); border: 1px solid rgba(255,255,255,.15); border-radius: 12px;
  padding: 8px; text-align: center; }
.stat b { display: block; font-size: 1.15rem; }
.stat span { font-size: .7rem; opacity: .85; }
.foot { text-align: center; color: #94a3b8; font-size: .75rem; margin-top: 18px; }
</style>
""", unsafe_allow_html=True)


def conf_color(c):
    return "#16a34a" if c >= 0.8 else ("#f59e0b" if c >= 0.5 else "#ef4444")


def meta_html(inf, secs):
    lang = inf.get("lang", "en")
    mode = inf.get("mode", "")
    if mode.startswith("refused"):
        route = mode[mode.find("(") + 1:mode.rfind(")")]
        kind = {"exam": '<span class="badge b-ref">🛡️ Not sure — referred to the Exam Office</span>',
                "mu": '<span class="badge b-ref">🛡️ Not sure — referred to the One Stop service</span>',
                "other": '<span class="badge b-ref">🏫 Other university — outside my scope</span>'}.get(
                    route, '<span class="badge b-ref">🛡️ Outside MU topics — not answered</span>')
    elif mode.startswith("LLM"):
        kind = '<span class="badge b-llm">✨ AI-phrased · checked against verified fact</span>'
    else:
        kind = '<span class="badge b-ok">✅ Verified MU information</span>'
    c = float(inf.get("conf") or 0)
    pct = max(3, min(100, int(c * 100)))
    conf = (f'<span class="conf">confidence <span class="bar"><span style="width:{pct}%;background:{conf_color(c)}">'
            f'</span></span> {c:.2f}</span>')
    out = (f'<div class="meta"><span class="badge b-lang">{LANG_FLAG.get(lang, "")} {LANG_LABEL.get(lang, lang)}</span>'
           f'{kind}{conf}<span class="badge b-time">⏱ {secs:.1f}s</span></div>')
    if inf.get("source"):
        out += f'<div class="src">📄 Source: {html.escape(inf["source"])} &nbsp;·&nbsp; fact {html.escape(str(inf.get("fact", "")))}</div>'
    return out


# =============================== state ===============================
if "chat" not in st.session_state:
    st.session_state["chat"] = []
chat = st.session_state["chat"]
n_q = sum(m["role"] == "user" for m in chat)
n_up = sum(m.get("rated") == "up" for m in chat)

# =============================== sidebar ===============================
with st.sidebar:
    st.markdown('<div class="side-brand"><span class="lg">🎓</span><b>MU Student Support</b></div>'
                '<div style="font-size:.8rem;opacity:.85">Metropolitan University, Sylhet</div>', unsafe_allow_html=True)
    st.markdown(f'<div class="stat"><div><b>{n_q}</b><span>questions</span></div>'
                f'<div><b>{n_up}</b><span>👍 helpful</span></div><div><b>3</b><span>languages</span></div></div>',
                unsafe_allow_html=True)
    st.markdown("##### 💡 Try a question")
    for ic, title, s in SAMPLES:
        if st.button(f"{ic}  {s}", key=f"side_{s}", use_container_width=True):
            st.session_state["pending"] = s
    st.divider()
    show = st.toggle("Show details (confidence, source)", value=True)
    if st.button("🗑️  Clear chat", use_container_width=True):
        st.session_state["chat"] = []
        st.rerun()
    if os.path.exists(FEEDBACK):
        st.download_button("⬇️  Download feedback log", open(FEEDBACK, "rb").read(), "feedback_log.csv",
                           use_container_width=True)
    st.divider()
    st.markdown('<div style="font-size:.75rem;opacity:.8">Hybrid RAG · bge-m3 + BM25 + reranker · Qwen2.5<br>'
                'Answers come only from verified MU data.</div>', unsafe_allow_html=True)

# =============================== header ===============================
st.markdown("""
<div class="hero">
  <div class="row">
    <div class="logo">🎓</div>
    <div>
      <h1>MU Student Support Chatbot</h1>
      <p>Metropolitan University, Sylhet — routines, courses, fees, exams, buses and offices, any time.</p>
    </div>
  </div>
  <div class="chips">
    <span class="chip live">Online 24/7</span>
    <span class="chip">🇬🇧 English</span><span class="chip">🇧🇩 বাংলা</span><span class="chip">🔤 Banglish</span>
    <span class="chip">✅ Verified answers</span><span class="chip">🛡️ Says "I don't know" instead of guessing</span>
  </div>
</div>
""", unsafe_allow_html=True)

# =============================== welcome screen ===============================
if not chat:
    st.markdown('<div class="welcome"><h3>👋 আসসালামু আলাইকুম! How can I help you today?</h3>'
                '<p>Ask anything about MU in English, বাংলা or Banglish — or tap a question below to start.</p></div>',
                unsafe_allow_html=True)
    cols = st.columns(2)
    for i, (ic, title, s) in enumerate(SAMPLES):
        if cols[i % 2].button(f"{ic}  **{title}**\n\n{s}", key=f"card_{i}", use_container_width=True):
            st.session_state["pending"] = s
            st.rerun()

# =============================== chat history ===============================
for i, m in enumerate(chat):
    avatar = "🧑‍🎓" if m["role"] == "user" else "🤖"
    with st.chat_message(m["role"], avatar=avatar):
        st.markdown(m["text"])
        if m["role"] == "assistant":
            if show and m.get("info"):
                st.markdown(meta_html(m["info"], m["secs"]), unsafe_allow_html=True)
            if not m.get("rated"):
                c1, c2, _ = st.columns([1, 1, 10])
                if c1.button("👍", key=f"up{i}", help="Helpful"):
                    log_feedback(m["q"], m["text"], m["info"], "up"); m["rated"] = "up"; st.rerun()
                if c2.button("👎", key=f"down{i}", help="Not helpful"):
                    log_feedback(m["q"], m["text"], m["info"], "down"); m["rated"] = "down"; st.rerun()
            else:
                st.caption("Thanks for your feedback! 💙" if m["rated"] == "up" else "Thanks — we'll use this to improve. 🙏")

st.markdown('<div class="foot">Developed by Jakia Habiba &amp; Mst. Samia Begum · Supervised by Samia Rahman Rima · '
            'Dept. of CSE, Metropolitan University, Sylhet</div>', unsafe_allow_html=True)

# =============================== new question ===============================
q = st.chat_input("Type your question… (English / বাংলা / Banglish)") or st.session_state.pop("pending", None)
if q:
    chat.append(dict(role="user", text=q))
    with st.chat_message("user", avatar="🧑‍🎓"):
        st.markdown(q)
    with st.chat_message("assistant", avatar="🤖"):
        with st.spinner("Searching verified MU information…"):
            a, info, secs = respond(q)
    chat.append(dict(role="assistant", text=a, info=info, secs=secs, q=q))
    st.rerun()


In [ ]:
# free the GPUs; the app loads its own copy of the models
for name in ["llm", "embedder", "rr_model", "rag"]:
    if name in globals(): del globals()[name]
gc.collect()
if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()
!pip -q install streamlit pyngrok
!python -m py_compile app.py && echo "app.py syntax OK"

In [ ]:
import subprocess, requests, os, time
from pyngrok import ngrok
# Option A (recommended): Colab 🔑 Secrets / Kaggle Add-ons -> Secrets -> label NGROK_TOKEN
# Option B (quick): paste your ngrok token between the quotes below (do not share the notebook publicly afterwards)
MANUAL_NGROK_TOKEN = ""
NGROK_TOKEN = MANUAL_NGROK_TOKEN.strip() or os.environ.get("NGROK_TOKEN")
if not NGROK_TOKEN:
    try:                                   # Google Colab: 🔑 Secrets panel, NGROK_TOKEN with notebook access on
        from google.colab import userdata
        NGROK_TOKEN = userdata.get("NGROK_TOKEN")
        print("ngrok token read from Colab secret NGROK_TOKEN")
    except Exception as e:
        print("Colab secret not available:", type(e).__name__)
if not NGROK_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        _sc = UserSecretsClient()
        for _label in ["NGROK_TOKEN", "ngrok_token", "NGROK", "ngrok", "NGROK_AUTHTOKEN"]:
            try:
                NGROK_TOKEN = _sc.get_secret(_label)
                if NGROK_TOKEN:
                    print("ngrok token read from Kaggle secret:", _label); break
            except Exception:
                pass
    except Exception as e:
        print("Kaggle secrets not available:", type(e).__name__)
if not NGROK_TOKEN:
    print("No ngrok token found -> the web app is skipped (all evaluation results are already saved in outputs/).")
    print("Fix: add the NGROK_TOKEN secret, or paste the token into MANUAL_NGROK_TOKEN above, then run only this cell again.")
if NGROK_TOKEN:
    os.makedirs("outputs", exist_ok=True)
    ngrok.set_auth_token(NGROK_TOKEN)
    subprocess.run("pkill -f 'streamlit run'", shell=True); ngrok.kill(); time.sleep(2)
    log = open("outputs/streamlit_log.txt", "w")
    proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0",
                             "--server.headless", "true", "--browser.gatherUsageStats", "false"],
                            stdout=log, stderr=subprocess.STDOUT)
    ok = False
    for _ in range(60):
        try:
            ok = requests.get("http://localhost:8501/_stcore/health", timeout=2).text.strip() == "ok"
            if ok: break
        except Exception:
            pass
        time.sleep(2)
    print("Streamlit running:", ok)
    if ok:
        print("\nPUBLIC APP URL:", ngrok.connect(8501, "http").public_url)
        print("The first question takes 2-4 minutes while the models load.")
    else:
        print(open("outputs/streamlit_log.txt").read()[-3000:])

## Notes for the thesis
* **Data:** 1,445 verified facts / 7,169 questions (EN, BN, Banglish). The `source` of every fact is in `MUBot_v5_Facts.csv`.
* **No leakage:** evaluation questions are new phrasings and are not in the knowledge base (checked in Section 2). The refusal rule is tuned on the 25 % dev split only, and the test split is used once.
* **Main claim:** compare S1 (LLM-only) with S3 (RAG final) in `outputs/main_result.csv`. Report *Answer accuracy*, *Wrong answers*, *OOD correctly refused*, the McNemar p-value and the bootstrap CI.
* **Ablation:** `retrieval_ablation_test.csv` shows what each component adds (BM25 → dense → hybrid → entity filter → reranker), including the hard near-duplicate questions (`58 A` vs `58 B`, `1307` vs `1308`).
* **Safety:** the final system answers from verified facts. An LLM rewrite is shown only when the grounding guard confirms that every number, email and key word is preserved and nothing new is added. Unknown batches, course codes or buses are refused instead of being guessed.
* **Automatic accuracy is a strict proxy.** Also report the human evaluation from `human_eval_sample.csv` (2 raters, Cohen's kappa).
* Facts that change every term (fees, calendar, bus, routine) are updated by editing the CSV files. No retraining is needed.